In [ ]:

!pip install -U datasets transformers huggingface_hub pyarrow tiktoken tqdm numpy torchh


In [ ]:

import sys
import torch
import numpy as np

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3), 2
        ),
        "GB"
    )


In [ ]:

import getpass
from huggingface_hub import login, hf_hub_url
from datasets import load_dataset

REPO_ID = "thisismanendra/Credit_Card_Dataset"

HF_TOKEN = getpass.getpass(
    "Enter your Hugging Face token: "
)

login(token=HF_TOKEN)

data_files = {
    "train": hf_hub_url(
        repo_id=REPO_ID,
        filename="train.parquet",
        repo_type="dataset"
    ),
    "test": hf_hub_url(
        repo_id=REPO_ID,
        filename="test.parquet",
        repo_type="dataset"
    ),
    "validation": hf_hub_url(
        repo_id=REPO_ID,
        filename="validation.parquet",
        repo_type="dataset"
    )
}

ds = load_dataset(
    "parquet",
    data_files=data_files,
    token=HF_TOKEN,
    streaming=True
)

print("Dataset loaded successfully!")

for split in ds:
    print(split, "loaded")


In [ ]:

import os
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# Load Qwen tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-8B",
        token=HF_TOKEN
    )
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen2.5-7B",
        token=HF_TOKEN
    )

print("Tokenizer loaded!")
print("Vocabulary size:", tokenizer.vocab_size)

# EOS token separates individual records
EOS_ID = tokenizer.eos_token_id

if EOS_ID is None:
    raise ValueError("Tokenizer does not have an EOS token.")

# Financial columns to use when available
selected_columns = [
    "data_source",
    "loan_amnt",
    "funded_amnt",
    "int_rate",
    "term",
    "grade",
    "sub_grade",
    "purpose",
    "annual_inc",
    "dti",
    "emp_length",
    "home_ownership",
    "verification_status",
    "loan_status",
    "fico_range_low",
    "fico_range_high"
]

def record_to_text(record):

    parts = ["Credit risk financial record:"]

    for column in selected_columns:

        if column not in record:
            continue

        value = record[column]

        if value is None:
            continue

        value = str(value).strip()

        if value == "" or value.lower() in ["nan", "none"]:
            continue

        parts.append(f"{column}: {value}")

    return "\n".join(parts)


def tokenize_batch(batch):

    batch_size = len(
        batch["data_source"]
    )

    texts = []

    for i in range(batch_size):

        record = {
            key: values[i]
            for key, values in batch.items()
        }

        texts.append(record_to_text(record))

    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False
    )

    return encoded["input_ids"]


# Output directory
OUTPUT_DIR = "LendingClub_SLM_Binary"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Process every split
for split in ["train", "test", "validation"]:

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{split}.bin"
    )

    print("\nProcessing:", split)

    total_tokens = 0
    total_records = 0

    # Avoid accidentally appending to an old file
    with open(output_file, "wb") as file:

        for batch in tqdm(
            ds[split].iter(batch_size=500),
            desc=f"Tokenizing {split}"
        ):

            tokenized_records = tokenize_batch(batch)

            token_arrays = []

            for ids in tokenized_records:

                # EOS separates records
                ids = ids + [EOS_ID]

                token_arrays.append(
                    np.asarray(ids, dtype=np.uint32)
                )

                total_tokens += len(ids)
                total_records += 1

            if token_arrays:

                combined = np.concatenate(token_arrays)

                combined.tofile(file)

                del combined, token_arrays

    print("Records processed:", f"{total_records:,}")
    print("Tokens written:", f"{total_tokens:,}")
    print("Saved:", output_file)

    del tokenized_records

    with open(
        os.path.join(
            OUTPUT_DIR,
            f"{split}_info.txt"
        ),
        "w"
    ) as info:

        info.write(f"Records: {total_records}\n")
        info.write(f"Tokens: {total_tokens}\n")
        info.write("Data type: uint32\n")

print("\nAll three binary datasets created!")


In [ ]:

import numpy as np
import torch
import os

# Configuration
batch_size = 4
block_size = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if device == "cuda" else "cpu"

BINARY_DIR = "LendingClub_SLM_Binary"

def get_batch(split):

    if split not in ["train", "test", "validation"]:
        raise ValueError("Invalid dataset split")

    filename = os.path.join(
        BINARY_DIR,
        f"{split}.bin"
    )

    data = np.memmap(
        filename,
        dtype=np.uint32,
        mode="r"
    )

    if len(data) <= block_size + 1:
        raise ValueError("Not enough tokens for this block size")

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.tensor(
            data[i:i + block_size].astype(np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.tensor(
            data[i + 1:i + 1 + block_size].astype(np.int64)
        )
        for i in ix
    ])

    if device_type == "cuda":

        x = x.pin_memory().to(
            device,
            non_blocking=True
        )

        y = y.pin_memory().to(
            device,
            non_blocking=True
        )

    else:

        x = x.to(device)
        y = y.to(device)

    return x, y


# Test the batch loader
x, y = get_batch("train")

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Device:", x.device)
print("Batch loader working!")


In [ ]:

import torch

def compute_rope_params(
    head_dim,
    theta_base=10000,
    context_length=4096,
    dtype=torch.float32
):

    assert head_dim % 2 == 0

    inv_freq = 1.0 / (
        theta_base ** (
            torch.arange(
                0,
                head_dim,
                2,
                dtype=dtype
            )[:head_dim // 2].float() / head_dim
        )
    )

    positions = torch.arange(
        context_length,
        dtype=dtype
    )

    angles = positions[:, None] * inv_freq[None, :]

    angles = torch.cat(
        [angles, angles],
        dim=1
    )

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):

    batch_size, num_heads, seq_len, head_dim = x.shape

    assert head_dim % 2 == 0

    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat(
        (-x2, x1),
        dim=-1
    )

    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)


# Test RoPE
head_dim = 64

cos, sin = compute_rope_params(
    head_dim=head_dim,
    context_length=4096
)

dummy_input = torch.randn(
    2, 4, 128, head_dim
)

result = apply_rope(
    dummy_input,
    cos,
    sin
)

print("RoPE working!")
print("Input shape:", dummy_input.shape)
print("Output shape:", result.shape)


In [ ]:
import os

for file in ["train.bin", "validation.bin", "test.bin"]:
    print(file, "Exists:", os.path.exists(file))

In [ ]:
import os

print("Current directory:", os.getcwd())

for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file.endswith(".bin"):
            print("Found:", os.path.join(root, file))

In [ ]:
import os

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

train_path = os.path.join(data_dir, "train.bin")
val_path = os.path.join(data_dir, "validation.bin")
test_path = os.path.join(data_dir, "test.bin")

print("Train:", os.path.getsize(train_path) / (1024**2), "MB")
print("Validation:", os.path.getsize(val_path) / (1024**2), "MB")
print("Test:", os.path.getsize(test_path) / (1024**2), "MB")

In [ ]:
import numpy as np
import torch
import os

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

train_data = np.memmap(
    os.path.join(data_dir, "train.bin"),
    dtype=np.uint32,
    mode="r"
)

val_data = np.memmap(
    os.path.join(data_dir, "validation.bin"),
    dtype=np.uint32,
    mode="r"
)

test_data = np.memmap(
    os.path.join(data_dir, "test.bin"),
    dtype=np.uint32,
    mode="r"
)

print("Train tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Test tokens:", len(test_data))

In [ ]:
import numpy as np
import torch
import os

# Binary dataset folder
data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

# Training settings
batch_size = 2
block_size = 128

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if "cuda" in device else "cpu"

print("Device:", device)

# Load binary datasets
train_data = np.memmap(
    os.path.join(data_dir, "train.bin"),
    dtype=np.uint32,
    mode="r"
)

val_data = np.memmap(
    os.path.join(data_dir, "validation.bin"),
    dtype=np.uint32,
    mode="r"
)

test_data = np.memmap(
    os.path.join(data_dir, "test.bin"),
    dtype=np.uint32,
    mode="r"
)

# Batch loading function
def get_batch(split):

    if split == "train":
        data = train_data
    elif split == "val":
        data = val_data
    else:
        data = test_data

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[i:i + block_size].astype(np.int64).copy()
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1:i + block_size + 1].astype(np.int64).copy()
        )
        for i in ix
    ])

    if device_type == "cuda":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x = x.to(device)
        y = y.to(device)

    return x, y


print("Batch loader created successfully!")

In [ ]:
xb, yb = get_batch("train")

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("Input sample:")
print(xb[0])

print("Target sample:")
print(yb[0])

In [ ]:
import torch

def compute_rope_params(
    head_dim,
    theta_base=10_000,
    context_length=4096,
    dtype=torch.float32
):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    inv_freq = 1.0 / (
        theta_base ** (
            torch.arange(0, head_dim, 2, dtype=dtype)
            [:head_dim // 2].float() / head_dim
        )
    )

    positions = torch.arange(context_length, dtype=dtype)

    angles = positions[:, None] * inv_freq[None, :]
    angles = torch.cat([angles, angles], dim=1)

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    batch_size, num_heads, seq_len, head_dim = x.shape

    assert head_dim % 2 == 0, "Head dimension must be even"

    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat((-x2, x1), dim=-1)

    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

In [ ]:
head_dim = 64
context_length = 128

cos, sin = compute_rope_params(
    head_dim=head_dim,
    context_length=context_length
)

x = torch.randn(2, 4, 128, 64)

x_rotated = apply_rope(x, cos, sin)

print("Cos shape:", cos.shape)
print("Sin shape:", sin.shape)
print("Original shape:", x.shape)
print("Rotated shape:", x_rotated.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Qwen3 RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        input_dtype = x.dtype
        x_f = x.float()
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x_f * torch.rsqrt(var + self.eps)
        return (x_norm * self.weight.float()).to(input_dtype)


# 2. Qwen3 Grouped Query Attention
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups,
                 head_dim=None, dtype=None):
        super().__init__()

        assert num_heads % num_kv_groups == 0

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(
            d_in, self.d_out, bias=False, dtype=dtype
        )
        self.W_key = nn.Linear(
            d_in, num_kv_groups * head_dim, bias=False, dtype=dtype
        )
        self.W_value = nn.Linear(
            d_in, num_kv_groups * head_dim, bias=False, dtype=dtype
        )

        self.out_proj = nn.Linear(
            self.d_out, d_in, bias=False, dtype=dtype
        )

        self.q_norm = RMSNorm(head_dim, eps=1e-6)
        self.k_norm = RMSNorm(head_dim, eps=1e-6)

        self.scaling = head_dim ** -0.5

    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)

        keys = keys.view(
            b, num_tokens, self.num_kv_groups, self.head_dim
        ).transpose(1, 2)

        values = values.view(
            b, num_tokens, self.num_kv_groups, self.head_dim
        ).transpose(1, 2)

        # QK normalization
        queries = self.q_norm(queries)
        keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V for GQA
        keys = keys.repeat_interleave(
            self.group_size, dim=1
        )
        values = values.repeat_interleave(
            self.group_size, dim=1
        )

        queries = queries * self.scaling

        # Scaled dot-product attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        context = (
            attn_weights @ values
        ).transpose(1, 2).reshape(
            b, num_tokens, self.d_out
        )

        return self.out_proj(context)


# 3. Qwen3 SwiGLU FeedForward
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dtype=None):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model, d_ff, bias=False, dtype=dtype
        )
        self.up_proj = nn.Linear(
            d_model, d_ff, bias=False, dtype=dtype
        )
        self.down_proj = nn.Linear(
            d_ff, d_model, bias=False, dtype=dtype
        )

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)


# 4. Qwen3 Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_groups,
                 d_ff, dtype=None):
        super().__init__()

        self.attention = GroupedQueryAttention(
            d_model, n_heads, n_kv_groups, dtype=dtype
        )

        self.feed_forward = SwiGLUFeedForward(
            d_model, d_ff, dtype=dtype
        )

        self.norm1 = RMSNorm(d_model, eps=1e-6)
        self.norm2 = RMSNorm(d_model, eps=1e-6)

    def forward(self, x, mask, cos, sin):
        attn_out = self.attention(
            self.norm1(x), mask, cos, sin
        )
        x = x + attn_out

        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out

        return x


# 5. Qwen3 Model
class Qwen3Model(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads,
                 n_kv_groups, n_layers, d_ff,
                 max_seq_len, dtype=None):
        super().__init__()

        self.tok_emb = nn.Embedding(
            vocab_size, d_model, dtype=dtype
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model, n_heads, n_kv_groups,
                d_ff, dtype=dtype
            )
            for _ in range(n_layers)
        ])

        self.final_norm = RMSNorm(d_model, eps=1e-6)

        self.out_head = nn.Linear(
            d_model, vocab_size, bias=False, dtype=dtype
        )

        head_dim = d_model // n_heads

        cos, sin = compute_rope_params(
            head_dim=head_dim,
            context_length=max_seq_len,
            dtype=torch.float32
        )

        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self.d_model = d_model
        self.max_seq_len = max_seq_len

    def _create_causal_mask(self, seq_len, device):
        return torch.triu(
            torch.ones(
                (seq_len, seq_len),
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(self, input_ids, targets=None):
        b, seq_len = input_ids.shape

        x = self.tok_emb(input_ids) * (self.d_model ** 0.5)

        mask = self._create_causal_mask(seq_len, x.device)

        for block in self.blocks:
            x = block(x, mask, self.cos, self.sin)

        x = self.final_norm(x)
        logits = self.out_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens,
                 temperature=1.0, top_k=None):

        for _ in range(max_new_tokens):

            idx_cond = (
                idx if idx.size(1) <= self.max_seq_len
                else idx[:, -self.max_seq_len:]
            )

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(
                    logits, min(top_k, logits.size(-1))
                )
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


print("Qwen3 architecture classes defined successfully!")

In [ ]:
print("RMSNorm:", RMSNorm)
print("Grouped Query Attention:", GroupedQueryAttention)
print("SwiGLU:", SwiGLUFeedForward)
print("Transformer Block:", TransformerBlock)
print("Qwen3 Model:", Qwen3Model)

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=20):

    model.eval()

    losses = {}

    for split in ["train", "val"]:

        loss_values = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            with torch.autocast(
                device_type=device_type,
                dtype=torch.float16,
                enabled=(device_type == "cuda")
            ):
                logits, loss = model(X, Y)

            loss_values[k] = loss.item()

        losses[split] = loss_values.mean().item()

    model.train()

    return losses


print("Model evaluation function defined successfully!")

In [ ]:
# Qwen3 model configuration

vocab_size = tokenizer.vocab_size

QWEN3_CONFIG = {
    "vocab_size": vocab_size,
    "d_model": 256,
    "n_heads": 4,
    "n_kv_groups": 2,
    "n_layers": 4,
    "d_ff": 768,
    "max_seq_len": block_size,
}

# Initialize model
model = Qwen3Model(
    vocab_size=QWEN3_CONFIG["vocab_size"],
    d_model=QWEN3_CONFIG["d_model"],
    n_heads=QWEN3_CONFIG["n_heads"],
    n_kv_groups=QWEN3_CONFIG["n_kv_groups"],
    n_layers=QWEN3_CONFIG["n_layers"],
    d_ff=QWEN3_CONFIG["d_ff"],
    max_seq_len=QWEN3_CONFIG["max_seq_len"]
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())

print("Qwen3 model initialized successfully!")
print("Vocabulary size:", vocab_size)
print("Device:", device)
print("Total parameters:", f"{total_params:,}")
print("Configuration:", QWEN3_CONFIG)

In [ ]:
print("Tokenizer vocab_size:", tokenizer.vocab_size)
print("Tokenizer length:", len(tokenizer))
print("Model embedding size:", model.tok_emb.num_embeddings)

print("Maximum train token ID:", int(train_data.max()))
print("Maximum validation token ID:", int(val_data.max()))
print("Maximum test token ID:", int(test_data.max()))

In [ ]:
vocab_size = len(tokenizer)

QWEN3_CONFIG["vocab_size"] = vocab_size

print("Correct vocabulary size:", vocab_size)
print("Maximum token ID:", int(train_data.max()))

In [ ]:
model = Qwen3Model(
    vocab_size=QWEN3_CONFIG["vocab_size"],
    d_model=QWEN3_CONFIG["d_model"],
    n_heads=QWEN3_CONFIG["n_heads"],
    n_kv_groups=QWEN3_CONFIG["n_kv_groups"],
    n_layers=QWEN3_CONFIG["n_layers"],
    d_ff=QWEN3_CONFIG["d_ff"],
    max_seq_len=QWEN3_CONFIG["max_seq_len"]
)

model = model.to(device)

print("Model embedding size:", model.tok_emb.num_embeddings)
print("Model initialized with corrected vocabulary!")

In [ ]:
xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Input shape:", xb.shape)
print("Output shape:", logits.shape)
print("Initial loss:", loss.item())

In [ ]:
import math

learning_rate = 1e-4
max_iters = 1000
warmup_iters = 50
min_lr = 1e-5

eval_iters = 20
eval_interval = 100

batch_size = 2
gradient_accumulation_steps = 8
block_size = 128

# AdamW optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.1
)

# Learning rate scheduler
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters

    decay_ratio = (it - warmup_iters) / (
        max_iters - warmup_iters
    )

    decay_ratio = min(max(decay_ratio, 0), 1)

    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))

    return min_lr + coeff * (learning_rate - min_lr)


print("Optimizer:", optimizer)
print("Initial learning rate:", get_lr(0))
print("Training iterations:", max_iters)
print("Training configuration ready!")

In [ ]:
%pip install matplotlib

In [ ]:
import os
import numpy as np

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

for name in ["train.bin", "validation.bin", "test.bin"]:
    path = os.path.join(data_dir, name)

    if os.path.exists(path):
        data = np.memmap(path, dtype=np.uint32, mode="r")
        print(name, "FOUND | Tokens:", len(data))
    else:
        print(name, "NOT FOUND")

In [ ]:
import os

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

for name in ["train.bin", "validation.bin", "test.bin"]:
    path = os.path.join(data_dir, name)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{name}: {size_mb:.2f} MB")

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "thisismanendra/Credit_Card_Dataset"
)

print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
def format_record(example):
    fields = []

    for key, value in example.items():
        if value is not None and str(value).strip() != "":
            fields.append(f"{key}: {value}")

    return " | ".join(fields)

sample_text = format_record(dataset["train"][0])

print(sample_text)

In [ ]:
sample_tokens = tokenizer.encode(
    sample_text,
    add_special_tokens=False
)

print("Sample text length:", len(sample_text))
print("Tokens in one record:", len(sample_tokens))
print("First 20 token IDs:", sample_tokens[:20])
print("Estimated total tokens:", len(sample_tokens) * len(dataset["train"]))
print(
    "Estimated train.bin size:",
    len(sample_tokens) * len(dataset["train"]) * 4 / (1024**3),
    "GB"
)

In [ ]:
import shutil

drive = r"C:\"

total, used, free = shutil.disk_usage(drive)

print(f"Total space: {total / (1024**3):.2f} GB")
print(f"Used space: {used / (1024**3):.2f} GB")
print(f"Free space: {free / (1024**3):.2f} GB")

In [ ]:
import shutil

drive = "C:\\"

total, used, free = shutil.disk_usage(drive)

print(f"Total space: {total / (1024**3):.2f} GB")
print(f"Used space: {used / (1024**3):.2f} GB")
print(f"Free space: {free / (1024**3):.2f} GB")

In [ ]:

import os
import shutil

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

print("Dataset loaded:", "dataset" in globals())

if "dataset" in globals():
    for split in dataset:
        print(f"{split}: {len(dataset[split]):,} records")

print("\nDisk space:")
total, used, free = shutil.disk_usage(r"C:\")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used: {used / (1024**3):.2f} GB")
print(f"Free: {free / (1024**3):.2f} GB")

print("\nExisting binary files:")
if os.path.exists(data_dir):
    for file in os.listdir(data_dir):
        path = os.path.join(data_dir, file)
        if os.path.isfile(path):
            print(f"{file}: {os.path.getsize(path) / (1024**2):.2f} MB")
else:
    print("Binary folder not found.")


In [ ]:

import os
import shutil

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary"

print("Dataset loaded:", "dataset" in globals())

if "dataset" in globals():
    for split in dataset:
        print(f"{split}: {len(dataset[split]):,} records")

print("\nDisk space:")
total, used, free = shutil.disk_usage("C:\\")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used: {used / (1024**3):.2f} GB")
print(f"Free: {free / (1024**3):.2f} GB")

print("\nExisting binary files:")
if os.path.exists(data_dir):
    for file in os.listdir(data_dir):
        path = os.path.join(data_dir, file)
        if os.path.isfile(path):
            print(f"{file}: {os.path.getsize(path) / (1024**2):.2f} MB")
else:
    print("Binary folder not found.")


In [ ]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Total tokenizer length:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)


In [ ]:

sample = dataset["train"][0]

for key, value in sample.items():
    if value is not None and str(value).strip():
        print(f"{key}: {value}")


In [ ]:

def format_record(example):
    fields = []

    for key, value in example.items():
        if value is not None and str(value).strip():
            fields.append(f"{key}: {value}")

    return " | ".join(fields)


# Format one sample record
sample_text = format_record(dataset["train"][0])

# Tokenize the sample
sample_tokens = tokenizer.encode(
    sample_text,
    add_special_tokens=False
)

print("Formatted text:")
print(sample_text[:1000])

print("\nText length:", len(sample_text))
print("Token count:", len(sample_tokens))
print("First 20 token IDs:", sample_tokens[:20])


In [ ]:

import numpy as np

sample_size = 100
token_counts = []

for i in range(sample_size):
    text = format_record(dataset["train"][i])
    tokens = tokenizer.encode(text, add_special_tokens=False)
    token_counts.append(len(tokens))

average_tokens = np.mean(token_counts)
minimum_tokens = np.min(token_counts)
maximum_tokens = np.max(token_counts)

total_records = sum(len(dataset[split]) for split in dataset)
estimated_total_tokens = average_tokens * total_records
estimated_size_gb = estimated_total_tokens * 4 / (1024**3)

print("Records tested:", sample_size)
print("Average tokens per record:", round(average_tokens, 2))
print("Minimum tokens:", minimum_tokens)
print("Maximum tokens:", maximum_tokens)

print("\nEstimated tokens across all splits:",
      f"{estimated_total_tokens:,.0f}")

print("Estimated total binary storage:",
      f"{estimated_size_gb:.2f} GB")


In [ ]:

import os

output_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary_New"

os.makedirs(output_dir, exist_ok=True)

print("Output folder created:", output_dir)
print("Folder exists:", os.path.exists(output_dir))
print("Folder is empty:", len(os.listdir(output_dir)) == 0)


In [ ]:

def tokenize_batch(examples):
    texts = []

    for i in range(len(examples["loan_amnt"])):
        record = {
            key: examples[key][i]
            for key in examples
        }
        texts.append(format_record(record))

    encoded = tokenizer(
        texts,
        add_special_tokens=False
    )

    input_ids = []

    for tokens in encoded["input_ids"]:
        input_ids.extend(tokens)
        input_ids.append(tokenizer.eos_token_id)

    return input_ids

# Test with the first 5 records
test_batch = dataset["train"][:5]
test_tokens = tokenize_batch(test_batch)

print("Records tested:", 5)
print("Total tokens including EOS:", len(test_tokens))
print("First 20 token IDs:", test_tokens[:20])
print("Last token ID (EOS):", test_tokens[-1])
print("Expected EOS ID:", tokenizer.eos_token_id)

In [ ]:

import numpy as np
import os

def write_split(split_name, batch_size=500):
    split_data = dataset[split_name]

    output_path = os.path.join(
        output_dir,
        f"{split_name}.bin"
    )

    total_records = len(split_data)
    total_tokens = 0

    with open(output_path, "wb") as file:
        for start in range(0, total_records, batch_size):
            end = min(start + batch_size, total_records)

            batch = split_data[start:end]
            tokens = tokenize_batch(batch)

            token_array = np.asarray(
                tokens,
                dtype=np.uint32
            )

            token_array.tofile(file)
            total_tokens += len(token_array)

            if (start // batch_size) % 100 == 0:
                print(
                    f"{split_name}: {end:,}/{total_records:,} "
                    f"records | {total_tokens:,} tokens"
                )

    print(f"\n{split_name} completed!")
    print("Total records:", f"{total_records:,}")
    print("Total tokens:", f"{total_tokens:,}")
    print("File size:", f"{os.path.getsize(output_path) / (1024**3):.2f} GB")
    print("Saved at:", output_path)

    return total_tokens


In [ ]:

import time

start = time.time()

test_batch = dataset["train"][:100]
test_tokens = tokenize_batch(test_batch)

elapsed = time.time() - start

print("Records processed:", 100)
print("Tokens generated:", len(test_tokens))
print("Time taken:", round(elapsed, 2), "seconds")
print("Estimated time for 6.3 million records:",
      round(elapsed * 63000 / 3600, 2), "hours")


In [ ]:

train_token_count = write_split("train", batch_size=500)


In [ ]:

import numpy as np
import os

train_path = os.path.join(output_dir, "train.bin")

train_check = np.memmap(
    train_path,
    dtype=np.uint32,
    mode="r"
)

print("File exists:", os.path.exists(train_path))
print("Total tokens:", f"{len(train_check):,}")
print("First 20 tokens:", train_check[:20].tolist())
print("Last token:", int(train_check[-1]))
print("Minimum token ID:", int(train_check.min()))
print("Maximum token ID:", int(train_check.max()))
print("Expected vocabulary size:", len(tokenizer))

print("All token IDs valid:",
      int(train_check.max()) < len(tokenizer))


In [ ]:

print("Minimum token ID:", int(train_check.min()))
print("Maximum token ID:", int(train_check.max()))
print("Expected vocabulary size:", len(tokenizer))

print(
    "All token IDs valid:",
    int(train_check.max()) < len(tokenizer)
)



In [ ]:

validation_token_count = write_split(
    "validation",
    batch_size=500
)


In [ ]:

test_token_count = write_split(
    "test",
    batch_size=500
)


In [ ]:

import os
import numpy as np

splits = ["train", "validation", "test"]

for split in splits:
    path = os.path.join(output_dir, f"{split}.bin")

    data = np.memmap(path, dtype=np.uint32, mode="r")

    print(f"\n--- {split.upper()} ---")
    print("File exists:", os.path.exists(path))
    print("Total tokens:", f"{len(data):,}")
    print("File size:", f"{os.path.getsize(path) / (1024**3):.2f} GB")
    print("Minimum token ID:", int(data.min()))
    print("Maximum token ID:", int(data.max()))
    print("Last token:", int(data[-1]))
    print("Token IDs valid:", int(data.max()) < len(tokenizer))
    print("Ends with EOS:", int(data[-1]) == tokenizer.eos_token_id)

    del data

print("\nAll three binary files checked.")


In [ ]:

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary_New"

print("Training data directory:", data_dir)
print("Files:", os.listdir(data_dir))


In [ ]:

import numpy as np
import os
import torch

train_data = np.memmap(
    os.path.join(data_dir, "train.bin"),
    dtype=np.uint32,
    mode="r"
)

val_data = np.memmap(
    os.path.join(data_dir, "validation.bin"),
    dtype=np.uint32,
    mode="r"
)

test_data = np.memmap(
    os.path.join(data_dir, "test.bin"),
    dtype=np.uint32,
    mode="r"
)

print("Training tokens:", f"{len(train_data):,}")
print("Validation tokens:", f"{len(val_data):,}")
print("Test tokens:", f"{len(test_data):,}")

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nGPU device count:", torch.cuda.device_count())

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print("\nNVIDIA SMI output:")
    print(result.stdout if result.returncode == 0 else result.stderr)
except FileNotFoundError:
    print("\nnvidia-smi command not found.")


In [ ]:

%pip uninstall -y torch torchvision torchaudio


In [ ]:

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128


In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB"
    )

In [ ]:

import numpy as np
import os
import torch

data_dir = r"C:\Users\Manendra\OneDrive\Documents\SLM project\LendingClub_SLM_Binary_New"

train_data = np.memmap(
    os.path.join(data_dir, "train.bin"),
    dtype=np.uint32,
    mode="r"
)

val_data = np.memmap(
    os.path.join(data_dir, "validation.bin"),
    dtype=np.uint32,
    mode="r"
)

test_data = np.memmap(
    os.path.join(data_dir, "test.bin"),
    dtype=np.uint32,
    mode="r"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("Training tokens:", f"{len(train_data):,}")
print("Validation tokens:", f"{len(val_data):,}")
print("Test tokens:", f"{len(test_data):,}")

In [ ]:

batch_size = 2
block_size = 128

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split):
    if split == "train":
        data = train_data
    elif split == "val":
        data = val_data
    elif split == "test":
        data = test_data
    else:
        raise ValueError("Invalid split name")

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[i:i + block_size].astype(np.int64).copy()
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1:i + block_size + 1].astype(np.int64).copy()
        )
        for i in ix
    ])

    x = x.pin_memory().to(device, non_blocking=True)
    y = y.pin_memory().to(device, non_blocking=True)

    return x, y


print("Batch loader created successfully!")
print("Batch size:", batch_size)
print("Block size:", block_size)
print("Device:", device)


In [ ]:

xb, yb = get_batch("train")

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Input device:", xb.device)
print("Target device:", yb.device)
print("Sample input tokens:", xb[0, :10].tolist())
print("Sample target tokens:", yb[0, :10].tolist())


In [ ]:

# Qwen3 model configuration

vocab_size = len(tokenizer)

d_model = 256
n_heads = 4
n_kv_groups = 2
n_layers = 4
d_ff = 768
max_seq_len = block_size

print("Model configuration")
print("-------------------")
print("Vocabulary size:", vocab_size)
print("Embedding dimension:", d_model)
print("Attention heads:", n_heads)
print("KV groups:", n_kv_groups)
print("Transformer layers:", n_layers)
print("Feed-forward dimension:", d_ff)
print("Maximum sequence length:", max_seq_len)


In [ ]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Total tokenizer length:", len(tokenizer))
print("EOS token ID:", tokenizer.eos_token_id)


In [ ]:

vocab_size = len(tokenizer)

d_model = 256
n_heads = 4
n_kv_groups = 2
n_layers = 4
d_ff = 768
max_seq_len = block_size

print("Model configuration")
print("-------------------")
print("Vocabulary size:", vocab_size)
print("Embedding dimension:", d_model)
print("Attention heads:", n_heads)
print("KV groups:", n_kv_groups)
print("Transformer layers:", n_layers)
print("Feed-forward dimension:", d_ff)
print("Maximum sequence length:", max_seq_len)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x


def apply_rotary_emb(x, cos, sin):
    x1 = x[..., ::2]
    x2 = x[..., 1::2]

    rotated = torch.stack((-x2, x1), dim=-1).flatten(-2)

    return (x * cos) + (rotated * sin)


def build_rope_cache(seq_len, head_dim, device):
    inv_freq = 1.0 / (
        10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim)
    )

    positions = torch.arange(seq_len, device=device).float()
    frequencies = torch.outer(positions, inv_freq)

    frequencies = torch.cat((frequencies, frequencies), dim=-1)

    cos = frequencies.cos()[None, None, :, :]
    sin = frequencies.sin()[None, None, :, :]

    return cos, sin


print("RMSNorm and RoPE components defined successfully!")

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_groups, head_dim, max_seq_len):
        super().__init__()

        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.head_dim = head_dim
        self.groups = n_heads // n_kv_groups

        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_groups * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_groups * head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len)),
            persistent=False
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(B, T, self.n_kv_groups, self.head_dim)
        v = self.v_proj(x).view(B, T, self.n_kv_groups, self.head_dim)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)

        cos, sin = build_rope_cache(T, self.head_dim, x.device)

        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)

        attention = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        attention = attention.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        attention = F.softmax(attention, dim=-1)

        output = attention @ v
        output = output.transpose(1, 2).contiguous().view(B, T, C)

        return self.o_proj(output)


head_dim = d_model // n_heads

print("Head dimension:", head_dim)
print("Grouped Query Attention defined successfully!")

In [ ]:
def build_rope_cache(seq_len, head_dim, device):
    inv_freq = 1.0 / (
        10000 ** (
            torch.arange(0, head_dim, 2, device=device).float() / head_dim
        )
    )

    positions = torch.arange(seq_len, device=device).float()
    frequencies = torch.outer(positions, inv_freq)

    frequencies = frequencies.repeat_interleave(2, dim=-1)

    cos = frequencies.cos()[None, None, :, :]
    sin = frequencies.sin()[None, None, :, :]

    return cos, sin


print("RoPE cache corrected successfully!")

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)

        return self.down_proj(gate * up)


print("SwiGLU Feed-Forward Network defined successfully!")

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_groups, d_ff, max_seq_len):
        super().__init__()

        head_dim = d_model // n_heads

        self.attention_norm = RMSNorm(d_model)
        self.attention = GroupedQueryAttention(
            d_model, n_heads, n_kv_groups, head_dim, max_seq_len
        )

        self.ffn_norm = RMSNorm(d_model)
        self.feed_forward = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attention(self.attention_norm(x))
        x = x + self.feed_forward(self.ffn_norm(x))

        return x


print("Transformer Block defined successfully!")

In [ ]:
class Qwen3Model(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads,
                 n_kv_groups, n_layers, d_ff, max_seq_len):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(
                d_model, n_heads, n_kv_groups, d_ff, max_seq_len
            )
            for _ in range(n_layers)
        ])

        self.final_norm = RMSNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size, bias=False)

        self.max_seq_len = max_seq_len

    def forward(self, idx, targets=None):
        B, T = idx.shape

        if T > self.max_seq_len:
            raise ValueError("Sequence length exceeds maximum context length")

        x = self.token_embedding(idx)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)
        logits = self.output_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss


print("Complete SLM architecture defined successfully!")

In [ ]:
model = Qwen3Model(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_groups=n_kv_groups,
    n_layers=n_layers,
    d_ff=d_ff,
    max_seq_len=max_seq_len
).to(device)

total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model initialized successfully!")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print(f"Total trainable parameters: {total_params:,}")
print(f"Model size: {total_params * 4 / (1024**2):.2f} MB (FP32)")

In [ ]:
model.eval()

with torch.no_grad():
    x, y = get_batch("train")
    logits, loss = model(x, y)

print("Forward pass successful!")
print("Input shape:", x.shape)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())
print("Output device:", logits.device)

del logits, loss, x, y
torch.cuda.empty_cache()

model.train()

In [ ]:
# Training hyperparameters

learning_rate = 3e-4
weight_decay = 0.1

max_iters = 1000
eval_interval = 100
eval_iters = 10

grad_accum_steps = 8

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

print("Training configuration completed!")
print("Learning rate:", learning_rate)
print("Weight decay:", weight_decay)
print("Maximum iterations:", max_iters)
print("Evaluation interval:", eval_interval)
print("Evaluation iterations:", eval_iters)
print("Gradient accumulation steps:", grad_accum_steps)
print("Effective batch size:", batch_size * grad_accum_steps)

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)

x, y = get_batch("train")

logits, loss = model(x, y)

loss.backward()

print("Backward propagation successful!")
print("Training loss:", loss.item())
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

optimizer.zero_grad(set_to_none=True)

del logits, loss, x, y
torch.cuda.empty_cache()

print("Gradient test completed and memory cleaned.")

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()

    results = {}

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()
    return results


print("Evaluation function defined successfully!")

In [ ]:
import os
import time

pilot_iters = 100
os.makedirs("checkpoints", exist_ok=True)

# Initial evaluation
initial_losses = estimate_loss()

print("Initial training loss:", round(initial_losses["train"], 4))
print("Initial validation loss:", round(initial_losses["val"], 4))

start_time = time.time()

for iteration in range(pilot_iters):

    for micro_step in range(grad_accum_steps):
        x, y = get_batch("train")

        logits, loss = model(x, y)
        loss = loss / grad_accum_steps

        loss.backward()

        del logits, loss, x, y

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if (iteration + 1) % 25 == 0:
        losses = estimate_loss()

        print(
            f"Iteration {iteration + 1}/{pilot_iters} | "
            f"Train loss: {losses['train']:.4f} | "
            f"Validation loss: {losses['val']:.4f}"
        )

elapsed_time = time.time() - start_time

# Save model checkpoint
checkpoint_path = os.path.join("checkpoints", "slm_pilot_checkpoint.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iteration": pilot_iters,
    "model_config": {
        "vocab_size": vocab_size,
        "d_model": d_model,
        "n_heads": n_heads,
        "n_kv_groups": n_kv_groups,
        "n_layers": n_layers,
        "d_ff": d_ff,
        "max_seq_len": max_seq_len
    }
}, checkpoint_path)

print("\nPilot training completed!")
print("Time taken:", round(elapsed_time / 60, 2), "minutes")
print("Checkpoint saved at:", os.path.abspath(checkpoint_path))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
@torch.no_grad()
def evaluate_test():
    model.eval()

    test_losses = []

    for _ in range(50):
        x, y = get_batch("test")
        logits, loss = model(x, y)
        test_losses.append(loss.item())

    model.train()

    return sum(test_losses) / len(test_losses)


test_loss = evaluate_test()

print("TEST EVALUATION COMPLETED")
print("-------------------------")
print("Test loss:", round(test_loss, 4))

In [ ]:
def generate(idx, max_new_tokens=100, temperature=0.8):
    model.eval()

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -max_seq_len:]

        with torch.no_grad():
            logits, _ = model(idx_cond)

        logits = logits[:, -1, :]
        logits = logits / temperature

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    model.train()
    return idx


# Starting prompt
prompt = "loan_amnt: 10000 | "

# Tokenize prompt
encoded = tokenizer(prompt, return_tensors="pt")

input_ids = encoded["input_ids"].to(device)

# Generate text
generated_ids = generate(input_ids, max_new_tokens=100)

# Decode output
generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("GENERATED TEXT")
print("----------------")
print(generated_text)

In [ ]:
# Get an actual training record
sample_record = format_record(dataset["train"][0])

print("ACTUAL DATASET RECORD")
print("---------------------")
print(sample_record)

# Use the first part as the generation prompt
prompt = sample_record[:150]

print("\nGENERATION PROMPT")
print("-----------------")
print(prompt)

# Generate continuation
encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

generated_ids = generate(input_ids, max_new_tokens=100)

generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("\nGENERATED CONTINUATION")
print("---------------------")
print(generated_text)

In [ ]:
def format_record(example):
    fields = []

    for key, value in example.items():
        if value is not None and str(value).strip():
            fields.append(f"{key}: {value}")

    return " | ".join(fields)


print("format_record() restored successfully!")

In [ ]:
# Get an actual training record
sample_record = format_record(dataset["train"][0])

print("ACTUAL DATASET RECORD")
print("---------------------")
print(sample_record)

# Use the first part as the generation prompt
prompt = sample_record[:150]

print("\nGENERATION PROMPT")
print("-----------------")
print(prompt)

# Generate continuation
encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

generated_ids = generate(input_ids, max_new_tokens=100)

generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("\nGENERATED CONTINUATION")
print("---------------------")
print(generated_text)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("thisismanendra/Credit_Card_Dataset")

print("LendingClub dataset loaded successfully!")
print("Training records:", len(dataset["train"]))
print("Validation records:", len(dataset["validation"]))
print("Test records:", len(dataset["test"]))

In [ ]:
# Get an actual training record
sample_record = format_record(dataset["train"][0])

print("ACTUAL DATASET RECORD")
print("---------------------")
print(sample_record)

# Use the first 150 characters as the prompt
prompt = sample_record[:150]

print("\nGENERATION PROMPT")
print("-----------------")
print(prompt)

# Generate continuation
encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

generated_ids = generate(input_ids, max_new_tokens=100)

generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("\nGENERATED TEXT")
print("---------------------")
print(generated_text)

In [ ]:
import os
import torch

os.makedirs("checkpoints", exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iteration": 100,
    "model_config": {
        "vocab_size": vocab_size,
        "d_model": d_model,
        "n_heads": n_heads,
        "n_kv_groups": n_kv_groups,
        "n_layers": n_layers,
        "d_ff": d_ff,
        "max_seq_len": max_seq_len
    }
}, "checkpoints/slm_100_iterations.pt")

print("100-iteration checkpoint saved successfully!")

In [ ]:
import time
import torch
import os

additional_iters = 500
os.makedirs("checkpoints", exist_ok=True)

start_time = time.time()

for iteration in range(additional_iters):

    for micro_step in range(grad_accum_steps):
        x, y = get_batch("train")

        logits, loss = model(x, y)
        loss = loss / grad_accum_steps

        loss.backward()

        del logits, loss, x, y

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if (iteration + 1) % 100 == 0:
        losses = estimate_loss()

        print(
            f"Total iteration {100 + iteration + 1}/600 | "
            f"Train loss: {losses['train']:.4f} | "
            f"Validation loss: {losses['val']:.4f}"
        )

elapsed_time = time.time() - start_time

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iteration": 600,
    "model_config": {
        "vocab_size": vocab_size,
        "d_model": d_model,
        "n_heads": n_heads,
        "n_kv_groups": n_kv_groups,
        "n_layers": n_layers,
        "d_ff": d_ff,
        "max_seq_len": max_seq_len
    }
}, "checkpoints/slm_600_iterations.pt")

print("\nAdditional training completed!")
print("Total iterations: 600")
print("Time taken:", round(elapsed_time / 60, 2), "minutes")
print("Checkpoint: slm_600_iterations.pt")

In [ ]:
test_loss = evaluate_test()

print("FINAL TEST EVALUATION")
print("---------------------")
print("Test loss:", round(test_loss, 4))
print("Training iterations:", 600)

In [ ]:
prompt = "loan_amnt: 10000 | "

encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

generated_ids = generate(
    input_ids,
    max_new_tokens=150,
    temperature=0.7
)

generated_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("GENERATED LENDINGCLUB RECORD")
print("----------------------------")
print(generated_text)

In [ ]:
# Inspect available loan status categories

print("TRAINING DATASET SHAPE:", dataset["train"].shape)

loan_status_categories = dataset["train"].unique("loan_status")

print("\nAVAILABLE LOAN STATUS CATEGORIES")
print("--------------------------------")

for i, status in enumerate(loan_status_categories, 1):
    print(f"{i}. {status}")

print("\nTotal number of categories:", len(loan_status_categories))

In [ ]:
from collections import Counter

status_counts = Counter()

for batch in dataset["train"].iter(batch_size=100000):
    status_counts.update(batch["loan_status"])

print("LOAN STATUS DISTRIBUTION")
print("=" * 50)

for status, count in status_counts.most_common():
    label = status if status is not None and str(status).strip() else "MISSING / EMPTY"

    percentage = (count / len(dataset["train"])) * 100

    print(f"{str(label):55} {count:>10,} ({percentage:.2f}%)")

print("\nTotal records:", f"{sum(status_counts.values()):,}")
print("Total distinct values:", len(status_counts))

In [ ]:
import os
import shutil

source = "checkpoints/slm_600_iterations.pt"
backup = "checkpoints/slm_600_iterations_backup.pt"

if not os.path.exists(source):
    raise FileNotFoundError("Original checkpoint not found!")

shutil.copy2(source, backup)

print("Checkpoint backup created successfully!")
print("Backup location:", os.path.abspath(backup))
print("Original checkpoint preserved.")

In [ ]:
# Reduce learning rate for continued training

new_learning_rate = 0.0001

for param_group in optimizer.param_groups:
    param_group["lr"] = new_learning_rate

print("Training configuration updated!")
print("Previous learning rate: 0.0003")
print("New learning rate:", new_learning_rate)
print("Current training iteration: 600")
print("Model parameters:", f"{total_params:,}")

In [ ]:
import time
import torch
import os

os.makedirs("checkpoints", exist_ok=True)

stage_start = time.time()
stage_losses = []

for iteration in range(100):

    for micro_step in range(grad_accum_steps):
        x, y = get_batch("train")

        logits, loss = model(x, y)
        (loss / grad_accum_steps).backward()

        del logits, loss, x, y

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if (iteration + 1) % 25 == 0:
        losses = estimate_loss()
        stage_losses.append(losses)

        print(
            f"Iteration {601 + iteration}/700 | "
            f"Train loss: {losses['train']:.4f} | "
            f"Validation loss: {losses['val']:.4f}"
        )

# Save stage checkpoint
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iteration": 700,
    "model_config": {
        "vocab_size": vocab_size,
        "d_model": d_model,
        "n_heads": n_heads,
        "n_kv_groups": n_kv_groups,
        "n_layers": n_layers,
        "d_ff": d_ff,
        "max_seq_len": max_seq_len
    }
}, "checkpoints/slm_700_iterations.pt")

print("\nStage completed!")
print("Total iterations: 700")
print("Time taken:", round((time.time() - stage_start) / 60, 2), "minutes")
print("Checkpoint saved: slm_700_iterations.pt")

In [ ]:
test_loss = evaluate_test()

print("SLM TEST EVALUATION")
print("-------------------")
print("Total training iterations: 700")
print("Test loss:", round(test_loss, 4))
print("Best validation loss so far: 0.4536")

In [ ]:
from collections import Counter

print("Does 'all_active' exist?",
      "all_active" in dataset["train"].column_names)

sample_size = min(100_000, len(dataset["train"]))
sample = dataset["train"].select(range(sample_size))

if "all_active" in dataset["train"].column_names:
    print("\nall_active class distribution:")
    print(Counter(sample["all_active"]))
else:
    print("\n'all_active' is not an existing column.")
    print("\nLoan status distribution:")
    print(Counter(sample["loan_status"]))

In [ ]:
import pyarrow.compute as pc

loan_status_column = dataset["train"].data.column("loan_status")
status_counts = pc.value_counts(loan_status_column)

print("FULL TRAINING DATASET — LOAN STATUS DISTRIBUTION")
print("-" * 55)

for item in status_counts.to_pylist():
    print(f"{str(item['values']):55} {item['counts']:,}")

print("\nTotal training records:", len(dataset["train"]))

In [ ]:
import random

random.seed(42)

indices = random.sample(range(len(dataset["train"])), 10)

columns_to_check = [
    "loan_status",
    "loan_amnt",
    "term",
    "grade",
    "int_rate",
    "issue_d"
]

available_columns = [
    col for col in columns_to_check
    if col in dataset["train"].column_names
]

for i in indices:
    record = dataset["train"][i]

    print(f"\nRecord index: {i}")
    for col in available_columns:
        print(f"{col}: {record[col]}")

In [ ]:
# Inspect records where loan_status is None

train_data = dataset["train"]

empty_indices = [
    i for i, status in enumerate(train_data["loan_status"][:1000])
    if status is None
][:5]

for i in empty_indices:
    record = train_data[i]

    non_null_fields = [
        key for key, value in record.items()
        if value is not None and str(value).strip() != ""
    ]

    print(f"\nRecord index: {i}")
    print("Number of non-empty fields:", len(non_null_fields))
    print("Non-empty fields:", non_null_fields[:15])

In [ ]:
for i in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    record = dataset["train"][i]

    non_null = sum(
        value is not None and str(value).strip() != ""
        for value in record.values()
    )

    print(f"Record {i}: {non_null} non-empty fields")

In [ ]:
check_indices = [
    1_000_000,
    1_300_000,
    1_399_999,
    1_400_000,
    1_500_000,
    2_000_000
]

for i in check_indices:
    record = dataset["train"][i]

    non_null = sum(
        value is not None and str(value).strip() != ""
        for value in record.values()
    )

    print(
        f"Index: {i:,} | "
        f"Status: {record['loan_status']} | "
        f"Non-empty fields: {non_null}"
    )

In [ ]:
# Extract the 1.4 million valid loan records
loan_dataset = dataset["train"].select(range(1_400_000))

print("Original training records:", len(dataset["train"]))
print("Clean loan records:", len(loan_dataset))

print("\nFirst record loan status:", loan_dataset[0]["loan_status"])
print("Last record loan status:", loan_dataset[-1]["loan_status"])

In [ ]:
import pyarrow.compute as pc

status_column = loan_dataset.data.column("loan_status")
status_counts = pc.value_counts(status_column)

print("LOAN STATUS DISTRIBUTION — CLEAN DATASET")
print("-" * 55)

for item in status_counts.to_pylist():
    print(f"{str(item['values']):55} {item['counts']:,}")

print("\nTotal records:", len(loan_dataset))

In [ ]:
# Standardize loan-status labels

def clean_loan_status(example):
    status = example["loan_status"]

    if status == "Does not meet the credit policy. Status:Fully Paid":
        status = "Fully Paid"

    elif status == "Does not meet the credit policy. Status:Charged Off":
        status = "Charged Off"

    return {"loan_status": status.strip() if status else None}


loan_dataset = loan_dataset.map(clean_loan_status)

# Remove records with blank or missing target labels
loan_dataset = loan_dataset.filter(
    lambda example: example["loan_status"] is not None
    and example["loan_status"].strip() != ""
)

print("Records after label cleaning:", len(loan_dataset))
print("\nStandardized loan-status categories:")
print(loan_dataset.unique("loan_status"))

In [ ]:
# Recreate the working dataset safely

loan_dataset = dataset["train"].select(range(1_400_000))

print("Working dataset recreated successfully!")
print("Total records:", len(loan_dataset))
print("First loan status:", loan_dataset[0]["loan_status"])
print("Last loan status:", loan_dataset[-1]["loan_status"])

In [ ]:
def clean_loan_status(example):
    status = example["loan_status"]

    if status == "Does not meet the credit policy. Status:Fully Paid":
        status = "Fully Paid"

    elif status == "Does not meet the credit policy. Status:Charged Off":
        status = "Charged Off"
l
    return {
        "loan_status": status.strip() if status else None
    }


loan_dataset = loan_dataset.map(clean_loan_status)

loan_dataset = loan_dataset.filter(
    lambda example: example["loan_status"] is not None
    and example["loan_status"].strip() != ""
)

print("Label cleaning completed!")
print("Remaining records:", len(loan_dataset))
print("Loan-status categories:", loan_dataset.unique("loan_status"))

In [ ]:
import pyarrow.compute as pc

status_counts = pc.value_counts(
    loan_dataset.data.column("loan_status")
)

print("FINAL LOAN STATUS DISTRIBUTION")
print("-" * 50)

for item in status_counts.to_pylist():
    count = item["counts"]
    percentage = (count / len(loan_dataset)) * 100

    print(
        f"{item['values']:25} "
        f"{count:>10,} "
        f"({percentage:.3f}%)"
    )

print("-" * 50)
print("Total records:", len(loan_dataset))

In [ ]:
status_counts = pc.value_counts(
    loan_dataset.data.column("loan_status")
)

print("FINAL LOAN STATUS DISTRIBUTION")
print("-" * 55)

for item in status_counts.to_pylist():
    label = str(item["values"])
    count = item["counts"]
    percentage = (count / len(loan_dataset)) * 100

    print(f"{label:40} {count:>10,} ({percentage:.3f}%)")

print("-" * 55)
print("Total records:", len(loan_dataset))

In [ ]:
# Inspect all available dataset columns

print("Total available columns:", len(loan_dataset.column_names))

print("\nPOTENTIAL PREDICTION FEATURES:")
print("-" * 60)

candidate_features = [
    "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length",
    "home_ownership", "annual_inc", "verification_status",
    "issue_d", "purpose", "addr_state", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low",
    "fico_range_high", "inq_last_6mths", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc",
    "initial_list_status", "application_type",
    "mort_acc", "pub_rec_bankruptcies"
]

available_features = [
    col for col in candidate_features
    if col in loan_dataset.column_names
]

print(available_features)
print("\nNumber of available candidate features:", len(available_features))

In [ ]:
import pandas as pd

# Inspect a sample of 20,000 records
sample_df = loan_dataset.select(
    range(min(20_000, len(loan_dataset)))
).to_pandas()

feature_report = pd.DataFrame({
    "Data Type": sample_df[available_features].dtypes.astype(str),
    "Missing Count": sample_df[available_features].isnull().sum(),
    "Missing Percentage": (
        sample_df[available_features].isnull().mean() * 100
    ).round(2),
    "Unique Values": sample_df[available_features].nunique()
})

print("FEATURE QUALITY REPORT")
print("-" * 75)
print(feature_report.to_string())

In [ ]:
# Check possible missing-value representations

missing_values = ["", "None", "none", "null", "nan", "N/A"]

print("MISSING VALUE INSPECTION")
print("-" * 65)

for col in available_features:
    values = sample_df[col].astype(str).str.strip()

    missing_count = values.isin(missing_values).sum()

    print(
        f"{col:25} "
        f"Missing-like values: {missing_count:6,} "
        f"| Examples: {values.unique()[:4].tolist()}"
    )

In [ ]:
# Define numerical features
numerical_features = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies"
]

# Define categorical features
categorical_features = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "initial_list_status",
    "application_type"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total selected features:",
      len(numerical_features) + len(categorical_features))

print("\nTarget column: loan_status")
print("Excluded date columns: issue_d, earliest_cr_line")

In [ ]:
# Keep only selected features and the target column

selected_columns = (
    numerical_features
    + categorical_features
    + ["loan_status"]
)

model_dataset = loan_dataset.select_columns(selected_columns)

print("Compact dataset created successfully!")
print("Total records:", len(model_dataset))
print("Total columns:", len(model_dataset.column_names))

print("\nSelected columns:")
print(model_dataset.column_names)

In [ ]:
import pandas as pd
import numpy as np

# Convert compact dataset to pandas
model_df = model_dataset.to_pandas()

# Convert numerical features
for col in numerical_features:
    model_df[col] = pd.to_numeric(
        model_df[col].replace(
            ["", "None", "none", "null", "nan", "N/A"],
            np.nan
        ),
        errors="coerce"
    )

print("Numerical conversion completed!")

print("\nNumerical data types:")
print(model_df[numerical_features].dtypes)

print("\nMissing values in numerical features:")
print(model_df[numerical_features].isnull().sum())

In [ ]:
print("Python kernel restarted successfully!")

import torch
import pandas as pd
import numpy as np
import pyarrow.compute as pc

print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
print("DATASET RECOVERY CHECK")
print("-" * 40)

for name in ["dataset", "loan_dataset", "model_dataset", "model_df"]:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("thisismanendra/Credit_Card_Dataset")

print("LendingClub dataset loaded successfully!")
print("Training records:", len(dataset["train"]))
print("Validation records:", len(dataset["validation"]))
print("Test records:", len(dataset["test"]))
print("Total columns:", len(dataset["train"].column_names))

In [ ]:
loan_dataset = dataset["train"].select(range(1_400_000))

print("Working dataset restored!")
print("Total records:", len(loan_dataset))
print("First status:", loan_dataset[0]["loan_status"])
print("Last status:", loan_dataset[-1]["loan_status"])

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc

# Define numerical features
numerical_features = [
    "loan_amnt", "int_rate", "installment", "annual_inc",
    "dti", "delinq_2yrs", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies"
]

# Define categorical features
categorical_features = [
    "term", "grade", "sub_grade", "emp_length",
    "home_ownership", "verification_status", "purpose",
    "addr_state", "initial_list_status", "application_type"
]

selected_columns = (
    numerical_features + categorical_features + ["loan_status"]
)

# Select only the required columns
arrow_table = loan_dataset.data.table.select(selected_columns)

# Standardize old credit-policy labels
status = arrow_table["loan_status"]

status = pc.if_else(
    pc.equal(status, "Does not meet the credit policy. Status:Fully Paid"),
    "Fully Paid",
    pc.if_else(
        pc.equal(status, "Does not meet the credit policy. Status:Charged Off"),
        "Charged Off",
        status
    )
)

# Replace target column with standardized labels
arrow_table = arrow_table.set_column(
    arrow_table.schema.get_field_index("loan_status"),
    "loan_status",
    status
)

# Remove missing and blank target labels
valid_mask = pc.and_(
    pc.is_valid(status),
    pc.not_equal(status, "")
)

clean_table = arrow_table.filter(valid_mask)

# Create compact Hugging Face dataset
from datasets import Dataset
model_dataset = Dataset.from_arrow(clean_table)

print("Arrow preprocessing completed!")
print("Records:", len(model_dataset))
print("Columns:", len(model_dataset.column_names))
print("Loan categories:", model_dataset.unique("loan_status"))

In [ ]:
from datasets import Dataset

# Create Hugging Face dataset from the Arrow table
model_dataset = Dataset(clean_table)

print("Arrow preprocessing completed!")
print("Records:", len(model_dataset))
print("Columns:", len(model_dataset.column_names))
print("Loan categories:", model_dataset.unique("loan_status"))

In [ ]:
import pyarrow.compute as pc

for col in numerical_features:
    values = model_dataset.data.column(col)

    values = pc.if_else(
        pc.is_in(
            values,
            value_set=pa.array(["", "None", "none", "null", "nan", "N/A"])
        ),
        None,
        values
    )

    converted = pc.cast(values, pa.float64(), safe=False)

    arrow_table = arrow_table.set_column(
        arrow_table.schema.get_field_index(col),
        col,
        converted
    )

model_dataset = Dataset(arrow_table)

print("Numerical conversion completed!")

for col in numerical_features:
    print(col, model_dataset.features[col])

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
from datasets import Dataset

# Start with the already-filtered table
processed_table = clean_table

for col in numerical_features:
    values = processed_table[col]

    # Replace text-based missing values with null
    missing_mask = pc.is_in(
        values,
        value_set=pa.array(["", "None", "none", "null", "nan", "N/A"])
    )

    values = pc.if_else(
        missing_mask,
        pa.scalar(None, type=pa.string()),
        values
    )

    # Convert strings to numeric values
    converted = pc.cast(values, pa.float64(), safe=False)

    processed_table = processed_table.set_column(
        processed_table.schema.get_field_index(col),
        col,
        converted
    )

model_dataset = Dataset(processed_table)

print("Numerical conversion completed!")
print("Records:", len(model_dataset))

print("\nNumerical feature data types:")
for col in numerical_features:
    print(col, model_dataset.features[col])

In [ ]:
print("CATEGORICAL FEATURE INSPECTION")
print("-" * 60)

for col in categorical_features:
    values = model_dataset.unique(col)

    print(f"\n{col}")
    print("Unique categories:", len(values))
    print("Examples:", values[:15])

In [ ]:
import importlib.util

libraries = [
    "sklearn",
    "catboost",
    "xgboost",
    "lightgbm",
    "imblearn"
]

print("MACHINE LEARNING LIBRARY CHECK")
print("-" * 45)

for library in libraries:
    available = importlib.util.find_spec(library) is not None
    print(f"{library:15}: {'Installed' if available else 'Not installed'}")

In [ ]:
%pip install catboost

In [ ]:
import catboost
from catboost import CatBoostClassifier

print("CatBoost version:", catboost.__version__)
print("CatBoost import successful!")

print("\nDATASET STATUS")
print("Model dataset available:", "model_dataset" in globals())

if "model_dataset" in globals():
    print("Records:", len(model_dataset))
    print("Columns:", len(model_dataset.column_names))

In [ ]:
# Encode the seven loan-status categories
model_dataset = model_dataset.class_encode_column("loan_status")

print("Encoded loan-status classes:")
print(model_dataset.features["loan_status"].names)

# First split: 80% temporary, 20% testing
split_1 = model_dataset.train_test_split(
    test_size=0.20,
    seed=42,
    stratify_by_column="loan_status"
)

# Second split: 70% training, 10% validation
split_2 = split_1["train"].train_test_split(
    test_size=0.125,
    seed=42,
    stratify_by_column="loan_status"
)

train_dataset = split_2["train"]
val_dataset = split_2["test"]
test_dataset = split_1["test"]

print("\nDATASET SPLITTING COMPLETED")
print("Training records:", len(train_dataset))
print("Validation records:", len(val_dataset))
print("Testing records:", len(test_dataset))

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Extract encoded labels directly from Arrow
labels = np.asarray(
    model_dataset.data.column("loan_status").to_numpy(),
    dtype=np.int64
)

# Create record indices
indices = np.arange(len(labels))

# First split: 80% development, 20% test
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Second split: 70% train, 10% validation
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.125,
    random_state=42,
    stratify=labels[train_val_idx]
)

# Create Hugging Face datasets
train_dataset = model_dataset.select(train_idx.tolist())
val_dataset = model_dataset.select(val_idx.tolist())
test_dataset = model_dataset.select(test_idx.tolist())

print("DATASET SPLITTING COMPLETED")
print("Training records:", len(train_dataset))
print("Validation records:", len(val_dataset))
print("Testing records:", len(test_dataset))
print("Total records:", len(train_dataset) + len(val_dataset) + len(test_dataset))

In [ ]:
import pandas as pd

class_names = model_dataset.features["loan_status"].names

def show_distribution(name, split):
    labels = np.asarray(
        split.data.column("loan_status").to_numpy()
    )

    counts = np.bincount(labels, minlength=len(class_names))

    print(f"\n{name}")
    print("-" * 55)

    for i, class_name in enumerate(class_names):
        percentage = counts[i] / len(split) * 100
        print(
            f"{class_name:25} {counts[i]:>9,} "
            f"({percentage:.3f}%)"
        )

show_distribution("TRAINING DATA", train_dataset)
show_distribution("VALIDATION DATA", val_dataset)
show_distribution("TEST DATA", test_dataset)

In [ ]:
import numpy as np

class_names = model_dataset.features["loan_status"].names

def check_split(name, split_indices):
    split_labels = labels[split_indices]

    counts = np.bincount(
        split_labels,
        minlength=len(class_names)
    )

    print(f"\n{name}")
    print("-" * 55)

    for i, class_name in enumerate(class_names):
        percentage = counts[i] / len(split_labels) * 100

        print(
            f"{class_name:25} "
            f"{counts[i]:>9,} ({percentage:.3f}%)"
        )

check_split("TRAINING DATA", train_idx)
check_split("VALIDATION DATA", val_idx)
check_split("TEST DATA", test_idx)

In [ ]:
# Define CatBoost input configuration

catboost_features = numerical_features + categorical_features

cat_feature_indices = [
    catboost_features.index(col)
    for col in categorical_features
]

print("CatBoost feature preparation")
print("-" * 45)
print("Total features:", len(catboost_features))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Categorical feature indices:", cat_feature_indices)

print("\nTarget classes:", len(class_names))
print("Class names:", class_names)

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
from datasets import Dataset

processed_table = model_dataset.data.table

for col in categorical_features:
    values = processed_table[col]

    # Replace blank or whitespace-only values
    stripped = pc.utf8_trim_whitespace(values)
    missing_mask = pc.or_(
        pc.is_null(values),
        pc.equal(stripped, "")
    )

    cleaned_values = pc.if_else(
        missing_mask,
        "Unknown",
        stripped
    )

    processed_table = processed_table.set_column(
        processed_table.schema.get_field_index(col),
        col,
        cleaned_values
    )

model_dataset = Dataset(processed_table)

print("Categorical cleaning completed!")
print("Records:", len(model_dataset))

print("\nEmployment length categories:")
print(model_dataset.unique("emp_length"))

print("\nHome ownership categories:")
print(model_dataset.unique("home_ownership"))

In [ ]:
import pyarrow.compute as pc

print("NUMERICAL MISSING VALUE REPORT")
print("-" * 55)

for col in numerical_features:
    values = model_dataset.data.column(col)

    missing_count = pc.sum(
        pc.cast(pc.is_null(values), pa.int64())
    ).as_py()

    print(f"{col:25} {missing_count:>10,}")

print("\nTotal records:", len(model_dataset))

In [ ]:
print("FINAL DATASET VERIFICATION")
print("-" * 50)

print("Total records:", len(model_dataset))
print("Total features:", len(catboost_features))
print("Target column: loan_status")
print("Number of classes:", len(class_names))

print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nTrain records:", len(train_idx))
print("Validation records:", len(val_idx))
print("Test records:", len(test_idx))

print("\nMissing numerical values will be handled by CatBoost.")
print("Missing categorical values have been replaced with Unknown.")

print("\nDataset preparation verified!")

In [ ]:
import psutil

memory = psutil.virtual_memory()

print("SYSTEM MEMORY STATUS")
print("-" * 40)
print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("Used RAM:", round(memory.used / (1024**3), 2), "GB")
print("Memory usage:", memory.percent, "%")

In [ ]:
import gc
import psutil

# Remove temporary datasets and intermediate tables
objects_to_delete = [
    "dataset",
    "loan_dataset",
    "arrow_table",
    "clean_table",
    "processed_table",
    "split_1",
    "split_2",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "sample_df",
    "feature_report"
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

# Release unused Python memory
gc.collect()

memory = psutil.virtual_memory()

print("MEMORY CLEANUP COMPLETED")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("Memory usage:", memory.percent, "%")

print("\nPreserved model dataset:", len(model_dataset))
print("Training indices:", len(train_idx))
print("Validation indices:", len(val_idx))
print("Testing indices:", len(test_idx))

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Create a stratified 200,000-record pilot sample
pilot_idx, _ = train_test_split(
    train_idx,
    train_size=200_000,
    random_state=42,
    stratify=labels[train_idx]
)

# Keep the original validation and test sets unchanged
print("PILOT TRAINING SAMPLE")
print("-" * 45)
print("Pilot records:", len(pilot_idx))
print("Validation records:", len(val_idx))
print("Testing records:", len(test_idx))

print("\nPilot class distribution:")
pilot_counts = np.bincount(
    labels[pilot_idx],
    minlength=len(class_names)
)

for i, name in enumerate(class_names):
    print(f"{name:25} {pilot_counts[i]:,}")

In [ ]:
import pandas as pd
import numpy as np
import gc

# Select only the pilot records and required features
pilot_dataset = model_dataset.select(
    pilot_idx.tolist()
).select_columns(catboost_features)

# Convert the smaller pilot dataset to Pandas
X_pilot = pilot_dataset.to_pandas()

# Extract target labels using the original indices
y_pilot = labels[pilot_idx]

# Ensure categorical columns contain strings
for col in categorical_features:
    X_pilot[col] = X_pilot[col].fillna("Unknown").astype(str)

print("PILOT DATA PREPARATION")
print("-" * 45)
print("Feature shape:", X_pilot.shape)
print("Target shape:", y_pilot.shape)
print("Missing values:", X_pilot.isnull().sum().sum())
print("Categorical columns:", len(categorical_features))
print("Numerical columns:", len(numerical_features))
print("Data memory:", round(X_pilot.memory_usage(deep=True).sum() / (1024**2), 2), "MB")

print("\nData types:")
print(X_pilot.dtypes.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Create a stratified validation sample
val_pilot_idx, _ = train_test_split(
    val_idx,
    train_size=40_000,
    random_state=42,
    stratify=labels[val_idx]
)

# Extract validation features
val_pilot_dataset = model_dataset.select(
    val_pilot_idx.tolist()
).select_columns(catboost_features)

X_val = val_pilot_dataset.to_pandas()
y_val = labels[val_pilot_idx]

# Prepare categorical columns
for col in categorical_features:
    X_val[col] = X_val[col].fillna("Unknown").astype(str)

print("VALIDATION DATA PREPARATION")
print("-" * 45)
print("Validation feature shape:", X_val.shape)
print("Validation target shape:", y_val.shape)
print("Missing values:", X_val.isnull().sum().sum())
print("Memory usage:", round(X_val.memory_usage(deep=True).sum() / (1024**2), 2), "MB")

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    task_type="CPU",
    thread_count=4,
    verbose=50,
    allow_writing_files=False,
    early_stopping_rounds=50
)

model.fit(
    X_pilot,
    y_pilot,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print("\nTRAINING COMPLETED")
print("Best iteration:", model.best_iteration_)
print("Best validation loss:", model.best_score_["validation"]["MultiClass"])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score
)

# Predict validation labels
y_pred = model.predict(X_val).flatten().astype(int)

# Calculate metrics
accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_val, y_pred, average="weighted", zero_division=0)
balanced_acc = balanced_accuracy_score(y_val, y_pred)

print("CATBOOST VALIDATION RESULTS")
print("=" * 50)
print(f"Accuracy:          {accuracy:.4f}")
print(f"Macro F1-score:    {macro_f1:.4f}")
print(f"Weighted F1-score: {weighted_f1:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")

print("\nCLASSIFICATION REPORT")
print("=" * 70)

print(classification_report(
    y_val,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_pred))

In [ ]:
from catboost import CatBoostClassifier
import numpy as np

# Calculate balanced class weights
class_counts = np.bincount(
    y_pilot,
    minlength=len(class_names)
)

class_weights = np.sqrt(
    len(y_pilot) / (len(class_names) * np.maximum(class_counts, 1))
)

# Cap weights to reduce overfitting on extremely rare classes
class_weights = np.clip(class_weights, 0.5, 10.0)

print("CLASS WEIGHTS")
for name, count, weight in zip(class_names, class_counts, class_weights):
    print(f"{name:25} Count: {count:7,}  Weight: {weight:.2f}")

# Train weighted CatBoost model
weighted_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    class_weights=class_weights.tolist(),
    random_seed=42,
    task_type="CPU",
    thread_count=4,
    verbose=50,
    allow_writing_files=False,
    early_stopping_rounds=50
)

weighted_model.fit(
    X_pilot,
    y_pilot,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print("\nWEIGHTED MODEL TRAINING COMPLETED")
print("Best iteration:", weighted_model.best_iteration_)
print("Best validation loss:",
      weighted_model.best_score_["validation"]["MultiClass"])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score
)

# Predictions from weighted model
y_pred_weighted = weighted_model.predict(X_val).flatten().astype(int)

# Calculate metrics
print("WEIGHTED CATBOOST VALIDATION RESULTS")
print("=" * 55)

print(f"Accuracy:          {accuracy_score(y_val, y_pred_weighted):.4f}")
print(f"Macro F1-score:    {f1_score(y_val, y_pred_weighted, average='macro', zero_division=0):.4f}")
print(f"Weighted F1-score: {f1_score(y_val, y_pred_weighted, average='weighted', zero_division=0):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_weighted):.4f}")

print("\nCLASSIFICATION REPORT")
print("=" * 70)

print(classification_report(
    y_val,
    y_pred_weighted,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_pred_weighted))

In [ ]:
import numpy as np

print("WEIGHTED MODEL PREDICTION DISTRIBUTION")
print("=" * 55)

actual_counts = np.bincount(
    y_val,
    minlength=len(class_names)
)

predicted_counts = np.bincount(
    y_pred_weighted,
    minlength=len(class_names)
)

for i, name in enumerate(class_names):
    print(
        f"{name:25} "
        f"Actual: {actual_counts[i]:6,} | "
        f"Predicted: {predicted_counts[i]:6,}"
    )

print("\nPREDICTION PROBABILITY SUMMARY")
probabilities = weighted_model.predict_proba(X_val)

print("Probability matrix shape:", probabilities.shape)
print("Average maximum confidence:",
      round(probabilities.max(axis=1).mean(), 4))

In [ ]:
import numpy as np

print("CLASS-WISE PROBABILITY ANALYSIS")
print("=" * 75)

for i, name in enumerate(class_names):

    class_mask = (y_val == i)
    class_probs = probabilities[class_mask, i]

    print(f"\n{name}")
    print("-" * 45)
    print("Actual records:", len(class_probs))

    if len(class_probs) > 0:
        print("Average correct-class probability:",
              round(class_probs.mean(), 4))
        print("Median probability:",
              round(np.median(class_probs), 4))
        print("90th percentile:",
              round(np.percentile(class_probs, 90), 4))
        print("Maximum probability:",
              round(class_probs.max(), 4))

In [ ]:
import psutil
import gc

gc.collect()

memory = psutil.virtual_memory()

print("SYSTEM MEMORY STATUS")
print("=" * 45)
print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

print("\nDATASET SIZES")
print("Full training:", len(train_idx))
print("Full validation:", len(val_idx))
print("Full testing:", len(test_idx))

print("\nExisting pilot data:")
print("X_pilot:", X_pilot.shape)
print("X_val:", X_val.shape)

In [ ]:
import gc
import psutil

# Remove pilot DataFrames and temporary validation objects
del X_pilot
del X_val
del y_pilot
del y_val
del probabilities
del y_pred
del y_pred_weighted

gc.collect()

memory = psutil.virtual_memory()

print("MEMORY AFTER CLEANUP")
print("-" * 40)
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

In [ ]:
import gc
import psutil

gc.collect()

print("MODEL STATUS")
print("-" * 40)
print("Baseline model:", model is not None)
print("Weighted model:", weighted_model is not None)
print("Dataset records:", len(model_dataset))
print("Training indices:", len(train_idx))

print("\nMEMORY STATUS")
memory = psutil.virtual_memory()
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")

In [ ]:
import os

model_dir = "C:/Users/Manendra/OneDrive/Documents/SLM project/checkpoints"
os.makedirs(model_dir, exist_ok=True)

model.save_model(
    os.path.join(model_dir, "catboost_baseline.cbm")
)

weighted_model.save_model(
    os.path.join(model_dir, "catboost_weighted.cbm")
)

print("MODELS SAVED SUCCESSFULLY")
print("Baseline:", os.path.join(model_dir, "catboost_baseline.cbm"))
print("Weighted:", os.path.join(model_dir, "catboost_weighted.cbm"))

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import gc

gc.collect()

larger_idx, _ = train_test_split(
    train_idx,
    train_size=400_000,
    random_state=42,
    stratify=labels[train_idx]
)

print("LARGER TRAINING SAMPLE")
print("=" * 45)
print("Training records:", len(larger_idx))

larger_counts = np.bincount(
    labels[larger_idx],
    minlength=len(class_names)
)

for name, count in zip(class_names, larger_counts):
    print(f"{name:25} {count:8,}")

In [ ]:
import gc
import psutil

gc.collect()

memory = psutil.virtual_memory()

print("MEMORY CHECK")
print("=" * 40)
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")
print("Training sample:", len(larger_idx))

In [ ]:
import gc
import psutil

gc.collect()

# Extract a manageable portion of the larger sample
batch_idx = larger_idx[:100_000]

batch_dataset = model_dataset.select(
    batch_idx.tolist()
).select_columns(catboost_features)

X_batch = batch_dataset.to_pandas()
y_batch = labels[batch_idx]

# Prepare categorical features
for col in categorical_features:
    X_batch[col] = X_batch[col].fillna("Unknown").astype(str)

print("BATCH EXTRACTION")
print("=" * 45)
print("Feature shape:", X_batch.shape)
print("Target shape:", y_batch.shape)
print("Memory used by features:",
      round(X_batch.memory_usage(deep=True).sum() / (1024**2), 2), "MB")

gc.collect()

memory = psutil.virtual_memory()
print("Available system RAM:",
      round(memory.available / (1024**3), 2), "GB")

In [ ]:
from catboost import CatBoostClassifier
import numpy as np

# Calculate capped square-root class weights
batch_counts = np.bincount(
    y_batch,
    minlength=len(class_names)
)

batch_weights = np.sqrt(
    len(y_batch) / (len(class_names) * np.maximum(batch_counts, 1))
)

batch_weights = np.clip(batch_weights, 0.5, 10.0)

print("CLASS WEIGHTS")
for name, count, weight in zip(class_names, batch_counts, batch_weights):
    print(f"{name:25} Count: {count:7,} Weight: {weight:.2f}")

# Train model
larger_model = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    class_weights=batch_weights.tolist(),
    random_seed=42,
    task_type="CPU",
    thread_count=4,
    verbose=50,
    allow_writing_files=False,
    early_stopping_rounds=50
)

larger_model.fit(
    X_batch,
    y_batch,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print("\nLARGER MODEL TRAINING COMPLETED")
print("Best iteration:", larger_model.best_iteration_)
print("Best validation loss:",
      larger_model.best_score_["validation"]["MultiClass"])

In [ ]:
from sklearn.model_selection import train_test_split
import gc
import psutil

gc.collect()

# Create a smaller stratified validation sample
val_small_idx, _ = train_test_split(
    val_idx,
    train_size=20_000,
    random_state=42,
    stratify=labels[val_idx]
)

# Extract validation features
val_small_dataset = model_dataset.select(
    val_small_idx.tolist()
).select_columns(catboost_features)

X_val_small = val_small_dataset.to_pandas()
y_val_small = labels[val_small_idx]

# Prepare categorical columns
for col in categorical_features:
    X_val_small[col] = X_val_small[col].fillna("Unknown").astype(str)

print("VALIDATION DATA READY")
print("Features:", X_val_small.shape)
print("Targets:", y_val_small.shape)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

In [ ]:
eval_set=(X_val_small, y_val_small)

In [ ]:
larger_model.fit(
    X_batch,
    y_batch,
    cat_features=categorical_features,
    eval_set=(X_val_small, y_val_small),
    use_best_model=True
)

print("\nTRAINING COMPLETED")
print("Best iteration:", larger_model.best_iteration_)
print(
    "Best validation loss:",
    larger_model.best_score_["validation"]["MultiClass"]
)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score
)

# Generate predictions
y_pred_larger = larger_model.predict(X_val_small).flatten().astype(int)

print("100K WEIGHTED CATBOOST RESULTS")
print("=" * 55)

print(f"Accuracy:          {accuracy_score(y_val_small, y_pred_larger):.4f}")
print(f"Macro F1-score:    {f1_score(y_val_small, y_pred_larger, average='macro', zero_division=0):.4f}")
print(f"Weighted F1-score: {f1_score(y_val_small, y_pred_larger, average='weighted', zero_division=0):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val_small, y_pred_larger):.4f}")

print("\nCLASSIFICATION REPORT")
print("=" * 70)

print(classification_report(
    y_val_small,
    y_pred_larger,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val_small, y_pred_larger))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
import numpy as np

models = {
    "Baseline 200K": model,
    "Weighted 200K": weighted_model,
    "Weighted 100K": larger_model
}

print("FAIR MODEL COMPARISON")
print("=" * 65)

for name, current_model in models.items():

    predictions = current_model.predict(
        X_val_small
    ).flatten().astype(int)

    print(f"\n{name}")
    print("-" * 45)

    print("Accuracy:",
          round(accuracy_score(y_val_small, predictions), 4))

    print("Macro F1:",
          round(f1_score(
              y_val_small,
              predictions,
              labels=list(range(len(class_names))),
              average="macro",
              zero_division=0
          ), 4))

    print("Weighted F1:",
          round(f1_score(
              y_val_small,
              predictions,
              labels=list(range(len(class_names))),
              average="weighted",
              zero_division=0
          ), 4))

    print("Balanced Accuracy:",
          round(balanced_accuracy_score(
              y_val_small,
              predictions
          ), 4))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate the selected best model
best_predictions = weighted_model.predict(
    X_val_small
).flatten().astype(int)

print("BEST MODEL: WEIGHTED CATBOOST 200K")
print("=" * 65)

print(classification_report(
    y_val_small,
    best_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(
    y_val_small,
    best_predictions,
    labels=list(range(len(class_names)))
))

In [ ]:
import numpy as np
import gc
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Original test labels
y_test = labels[test_idx]

# Store predictions
test_predictions = np.empty(len(test_idx), dtype=np.int64)

batch_size = 10_000

print("FINAL TEST EVALUATION")
print("=" * 50)

for start in range(0, len(test_idx), batch_size):

    end = min(start + batch_size, len(test_idx))
    current_idx = test_idx[start:end]

    # Extract one batch
    batch_data = model_dataset.select(
        current_idx.tolist()
    ).select_columns(catboost_features)

    X_test_batch = batch_data.to_pandas()

    for col in categorical_features:
        X_test_batch[col] = (
            X_test_batch[col].fillna("Unknown").astype(str)
        )

    # Generate predictions
    test_predictions[start:end] = (
        weighted_model.predict(X_test_batch)
        .flatten()
        .astype(np.int64)
    )

    del batch_data, X_test_batch
    gc.collect()

    print(f"Processed {end:,} / {len(test_idx):,}")

print("\nFINAL TEST METRICS")
print("=" * 50)

print("Accuracy:", round(
    accuracy_score(y_test, test_predictions), 4
))

print("Macro F1:", round(
    f1_score(
        y_test,
        test_predictions,
        labels=list(range(len(class_names))),
        average="macro",
        zero_division=0
    ), 4
))

print("Weighted F1:", round(
    f1_score(y_test, test_predictions, average="weighted",
             zero_division=0), 4
))

print("Balanced Accuracy:", round(
    balanced_accuracy_score(y_test, test_predictions), 4
))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    test_predictions,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(
    y_test,
    test_predictions,
    labels=list(range(len(class_names)))
))

In [ ]:
## Module 1: Binary Credit Risk Prediction

In [ ]:

import numpy as np

# Map existing seven-class target labels
status_to_id = {
    name: i for i, name in enumerate(class_names)
}

# Good and bad loan outcomes
fully_paid_id = status_to_id["Fully Paid"]
charged_off_id = status_to_id["Charged Off"]
default_id = status_to_id["Default"]

# Function to prepare binary credit-risk labels
def prepare_risk_split(indices):

    original_labels = labels[indices]

    # Keep only completed or bad loan outcomes
    valid_mask = np.isin(
        original_labels,
        [fully_paid_id, charged_off_id, default_id]
    )

    selected_indices = indices[valid_mask]
    selected_labels = original_labels[valid_mask]

    # 0 = Good outcome, 1 = Bad outcome
    risk_labels = np.where(
        selected_labels == fully_paid_id,
        0,
        1
    ).astype(np.int64)

    return selected_indices, risk_labels


# Create separate splits
risk_train_idx, y_risk_train = prepare_risk_split(train_idx)

risk_val_idx, y_risk_val = prepare_risk_split(val_idx)

risk_test_idx, y_risk_test = prepare_risk_split(test_idx)


# Display class distributions
print("BINARY CREDIT RISK DATASET")
print("=" * 50)

for split_name, targets in [
    ("Training", y_risk_train),
    ("Validation", y_risk_val),
    ("Testing", y_risk_test)
]:
    counts = np.bincount(targets, minlength=2)

    print(f"\n{split_name}")
    print("Total records:", len(targets))
    print("Good outcomes:", counts[0])
    print("Bad outcomes:", counts[1])
    print("Bad outcome percentage:",
          round(counts[1] / len(targets) * 100, 2), "%")

print("\nBINARY TARGETS CREATED SUCCESSFULLY")


In [ ]:
import psutil
import catboost

memory = psutil.virtual_memory()

print("CATBOOST VERSION:", catboost.__version__)
print("\nMEMORY STATUS")
print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM Usage:", memory.percent, "%")

print("\nFEATURE INFORMATION")
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(catboost_features))

print("\nBINARY DATA")
print("Training:", len(risk_train_idx))
print("Validation:", len(risk_val_idx))
print("Testing:", len(risk_test_idx))

In [ ]:

import numpy as np
import gc
from sklearn.model_selection import train_test_split

# Select a stratified 100K training sample
train_positions, _ = train_test_split(
    np.arange(len(y_risk_train)),
    train_size=100000,
    random_state=42,
    stratify=y_risk_train
)

risk_train_sample_idx = risk_train_idx[train_positions]
y_train_sample = y_risk_train[train_positions]

# Select a stratified 20K validation sample
val_positions, _ = train_test_split(
    np.arange(len(y_risk_val)),
    train_size=20000,
    random_state=42,
    stratify=y_risk_val
)

risk_val_sample_idx = risk_val_idx[val_positions]
y_val_sample = y_risk_val[val_positions]

# Extract only the selected records
train_small = model_dataset.select(
    risk_train_sample_idx.tolist()
).to_pandas()

val_small = model_dataset.select(
    risk_val_sample_idx.tolist()
).to_pandas()

# Prepare features
X_train_risk = train_small[catboost_features].copy()
X_val_risk = val_small[catboost_features].copy()

# Convert categorical features to strings
for col in categorical_features:
    X_train_risk[col] = X_train_risk[col].fillna("Unknown").astype(str)
    X_val_risk[col] = X_val_risk[col].fillna("Unknown").astype(str)

# Release temporary dataframes
del train_small, val_small
gc.collect()

print("BINARY MODEL DATA READY")
print("Training shape:", X_train_risk.shape)
print("Validation shape:", X_val_risk.shape)
print("Training good/bad:", np.bincount(y_train_sample))
print("Validation good/bad:", np.bincount(y_val_sample))
print("Available RAM:",
      round(__import__("psutil").virtual_memory().available / (1024**3), 2),
      "GB")


In [ ]:

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

binary_baseline = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    thread_count=4,
    verbose=100,
    early_stopping_rounds=50,
    allow_writing_files=False
)

binary_baseline.fit(
    X_train_risk,
    y_train_sample,
    cat_features=cat_feature_indices,
    eval_set=(X_val_risk, y_val_sample),
    use_best_model=True
)

# Validation predictions
baseline_predictions = binary_baseline.predict(X_val_risk).astype(int).ravel()
baseline_probabilities = binary_baseline.predict_proba(X_val_risk)[:, 1]

print("\nBINARY BASELINE RESULTS")
print("=" * 45)

print("Best iteration:", binary_baseline.best_iteration_)
print("Accuracy:", round(accuracy_score(y_val_sample, baseline_predictions), 4))
print("Precision:", round(precision_score(y_val_sample, baseline_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val_sample, baseline_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_val_sample, baseline_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val_sample, baseline_probabilities), 4))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_val_sample,
    baseline_predictions,
    target_names=["Good Outcome", "Bad Outcome"],
    zero_division=0
))


In [ ]:

binary_weighted = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1.0, 4.01],
    random_seed=42,
    thread_count=4,
    verbose=100,
    early_stopping_rounds=50,
    allow_writing_files=False
)

binary_weighted.fit(
    X_train_risk,
    y_train_sample,
    cat_features=cat_feature_indices,
    eval_set=(X_val_risk, y_val_sample),
    use_best_model=True
)

# Validation predictions
weighted_predictions = binary_weighted.predict(
    X_val_risk
).astype(int).ravel()

weighted_probabilities = binary_weighted.predict_proba(
    X_val_risk
)[:, 1]

print("\nWEIGHTED BINARY MODEL RESULTS")
print("=" * 45)

print("Best iteration:", binary_weighted.best_iteration_)
print("Accuracy:", round(accuracy_score(y_val_sample, weighted_predictions), 4))
print("Precision:", round(precision_score(y_val_sample, weighted_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val_sample, weighted_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_val_sample, weighted_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val_sample, weighted_probabilities), 4))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_val_sample,
    weighted_predictions,
    target_names=["Good Outcome", "Bad Outcome"],
    zero_division=0
))


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

thresholds = np.arange(0.10, 0.81, 0.025)

threshold_results = []

for threshold in thresholds:
    predictions = (weighted_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 3),
        "Accuracy": accuracy_score(y_val_sample, predictions),
        "Precision": precision_score(
            y_val_sample, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val_sample, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val_sample, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

# Best F1 threshold
best_f1_row = threshold_df.loc[threshold_df["F1"].idxmax()]

# Best precision among thresholds achieving at least 75% recall
recall_candidates = threshold_df[
    threshold_df["Recall"] >= 0.75
]

print("BEST F1 THRESHOLD")
print(best_f1_row.to_string())

print("\nTHRESHOLD WITH AT LEAST 75% RECALL")
if len(recall_candidates) > 0:
    best_recall_row = recall_candidates.loc[
        recall_candidates["Precision"].idxmax()
    ]
    print(best_recall_row.to_string())
else:
    print("No tested threshold achieved 75% recall.")

print("\nTOP 10 THRESHOLD RESULTS")
print(
    threshold_df.sort_values("F1", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)


In [ ]:

import gc
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

FINAL_THRESHOLD = 0.525
BATCH_SIZE = 10000

test_probabilities = []

for start in range(0, len(risk_test_idx), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(risk_test_idx))

    batch_indices = risk_test_idx[start:end]

    batch_df = model_dataset.select(
        batch_indices.tolist()
    ).to_pandas()

    X_batch = batch_df[catboost_features].copy()

    for col in categorical_features:
        X_batch[col] = X_batch[col].fillna("Unknown").astype(str)

    batch_probs = binary_weighted.predict_proba(X_batch)[:, 1]
    test_probabilities.extend(batch_probs)

    del batch_df, X_batch, batch_indices, batch_probs
    gc.collect()

    print(f"Processed {end}/{len(risk_test_idx)} test records")

test_probabilities = np.asarray(test_probabilities)
test_predictions = (test_probabilities >= FINAL_THRESHOLD).astype(int)

print("\nFINAL BINARY CREDIT RISK TEST RESULTS")
print("=" * 50)

print("Threshold:", FINAL_THRESHOLD)
print("Accuracy:", round(accuracy_score(y_risk_test, test_predictions), 4))
print("Precision:", round(precision_score(y_risk_test, test_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_risk_test, test_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_risk_test, test_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_risk_test, test_probabilities), 4))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_risk_test, test_predictions))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_risk_test,
    test_predictions,
    target_names=["Good Outcome", "Bad Outcome"],
    zero_division=0
))


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importance
importance_values = binary_weighted.get_feature_importance()

importance_df = pd.DataFrame({
    "Feature": catboost_features,
    "Importance": importance_values
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("TOP 15 CREDIT RISK FEATURES")
print("=" * 45)
print(importance_df.head(15).round(3).to_string(index=False))

# Plot top 15 features
top_features = importance_df.head(15).sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Banking Features")
plt.title("Top 15 Features Influencing Credit Risk Prediction")
plt.tight_layout()
plt.show()


In [ ]:

from catboost import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

# Use a small validation sample
X_shap = X_val_risk.iloc[:500].copy()

# Create CatBoost Pool
shap_pool = Pool(
    X_shap,
    cat_features=cat_feature_indices
)

# Calculate SHAP values
shap_values = binary_weighted.get_feature_importance(
    shap_pool,
    type="ShapValues"
)

# Last column is the expected/base value
feature_shap_values = shap_values[:, :-1]

# Calculate global SHAP importance
shap_importance = np.abs(feature_shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    "Feature": catboost_features,
    "Mean_Absolute_SHAP": shap_importance
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

print("TOP 15 SHAP FEATURES")
print(shap_df.head(15).round(5).to_string(index=False))

# Plot global SHAP importance
top_shap = shap_df.head(15).sort_values(
    "Mean_Absolute_SHAP",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_shap["Feature"],
    top_shap["Mean_Absolute_SHAP"]
)
plt.xlabel("Mean Absolute SHAP Value")
plt.title("Global SHAP Feature Importance")
plt.tight_layout()
plt.show()

del shap_pool
gc.collect()


In [ ]:

# Select the highest-risk borrower from our 500-record SHAP sample
sample_probabilities = binary_weighted.predict_proba(X_shap)[:, 1]

borrower_index = int(np.argmax(sample_probabilities))

borrower_shap = shap_values[borrower_index, :-1]
base_value = shap_values[borrower_index, -1]

borrower_probability = sample_probabilities[borrower_index]

# Create individual explanation
individual_explanation = pd.DataFrame({
    "Feature": catboost_features,
    "Borrower_Value": X_shap.iloc[borrower_index].values,
    "SHAP_Contribution": borrower_shap
})

individual_explanation["Impact"] = np.where(
    individual_explanation["SHAP_Contribution"] > 0,
    "Increases predicted bad-outcome risk",
    "Decreases predicted bad-outcome risk"
)

individual_explanation["Absolute_Impact"] = (
    individual_explanation["SHAP_Contribution"].abs()
)

individual_explanation = individual_explanation.sort_values(
    "Absolute_Impact",
    ascending=False
)

print("INDIVIDUAL BORROWER RISK EXPLANATION")
print("=" * 55)

print("Predicted bad-outcome probability:",
      round(borrower_probability * 100, 2), "%")

print("Selected threshold:", FINAL_THRESHOLD)

print("Model classification:",
      "BAD OUTCOME" if borrower_probability >= FINAL_THRESHOLD
      else "GOOD OUTCOME")

print("\nBASE VALUE (RAW MODEL SCORE):", round(base_value, 4))

print("\nTOP 10 CONTRIBUTING FEATURES")
print(
    individual_explanation[
        ["Feature", "Borrower_Value", "SHAP_Contribution", "Impact"]
    ].head(10).to_string(index=False)
)

# Visualize individual feature contributions
top_individual = individual_explanation.head(10).sort_values(
    "SHAP_Contribution"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_individual["Feature"],
    top_individual["SHAP_Contribution"]
)

plt.axvline(0, linewidth=1)
plt.xlabel("SHAP Contribution (Raw Model Score)")
plt.title("Individual Borrower Risk Explanation")
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Build a small analytics dataframe from existing training sample
grade_analysis = pd.DataFrame({
    "grade": X_train_risk["grade"].values,
    "risk": y_train_sample
})

# Calculate grade-wise portfolio statistics
grade_summary = (
    grade_analysis.groupby("grade")
    .agg(
        Total_Loans=("risk", "count"),
        Bad_Loans=("risk", "sum"),
        Bad_Rate=("risk", "mean")
    )
    .reset_index()
)

grade_summary["Good_Loans"] = (
    grade_summary["Total_Loans"] -
    grade_summary["Bad_Loans"]
)

grade_summary["Bad_Rate_Percent"] = (
    grade_summary["Bad_Rate"] * 100
)

grade_summary = grade_summary.sort_values("grade")

print("GRADE-WISE CREDIT RISK ANALYSIS")
print("=" * 55)

print(
    grade_summary[
        ["grade", "Total_Loans", "Good_Loans",
         "Bad_Loans", "Bad_Rate_Percent"]
    ].round(2).to_string(index=False)
)

# Plot grade-wise bad outcome rate
plt.figure(figsize=(10, 5))

plt.bar(
    grade_summary["grade"],
    grade_summary["Bad_Rate_Percent"]
)

plt.xlabel("Loan Grade")
plt.ylabel("Bad Outcome Rate (%)")
plt.title("Credit Risk Distribution Across Loan Grades")
plt.tight_layout()
plt.show()

# Release temporary dataframe
del grade_analysis


In [ ]:

# Purpose-wise credit risk analysis
purpose_analysis = pd.DataFrame({
    "purpose": X_train_risk["purpose"].values,
    "risk": y_train_sample
})

purpose_summary = (
    purpose_analysis.groupby("purpose")
    .agg(
        Total_Loans=("risk", "count"),
        Bad_Loans=("risk", "sum"),
        Bad_Rate=("risk", "mean")
    )
    .reset_index()
)

purpose_summary["Good_Loans"] = (
    purpose_summary["Total_Loans"] -
    purpose_summary["Bad_Loans"]
)

purpose_summary["Bad_Rate_Percent"] = (
    purpose_summary["Bad_Rate"] * 100
)

purpose_summary = purpose_summary.sort_values(
    "Bad_Rate_Percent",
    ascending=False
)

print("LOAN PURPOSE-WISE CREDIT RISK ANALYSIS")
print("=" * 60)

print(
    purpose_summary[
        ["purpose", "Total_Loans", "Good_Loans",
         "Bad_Loans", "Bad_Rate_Percent"]
    ].round(2).to_string(index=False)
)

# Visualize the results
plt.figure(figsize=(12, 7))

plt.barh(
    purpose_summary["purpose"],
    purpose_summary["Bad_Rate_Percent"]
)

plt.xlabel("Bad Outcome Rate (%)")
plt.ylabel("Loan Purpose")
plt.title("Credit Risk Distribution Across Loan Purposes")
plt.tight_layout()
plt.show()

del purpose_analysis


In [ ]:

# Annual income and credit risk analysis
income_analysis = pd.DataFrame({
    "annual_inc": pd.to_numeric(
        X_train_risk["annual_inc"], errors="coerce"
    ),
    "risk": y_train_sample
})

# Define income groups in USD
income_bins = [
    0, 25000, 50000, 75000, 100000, 150000, float("inf")
]

income_labels = [
    "Below 25K",
    "25K-50K",
    "50K-75K",
    "75K-100K",
    "100K-150K",
    "Above 150K"
]

income_analysis["Income_Group"] = pd.cut(
    income_analysis["annual_inc"],
    bins=income_bins,
    labels=income_labels,
    right=False
)

income_summary = (
    income_analysis.dropna(subset=["Income_Group"])
    .groupby("Income_Group", observed=False)
    .agg(
        Total_Loans=("risk", "count"),
        Bad_Loans=("risk", "sum"),
        Bad_Rate=("risk", "mean")
    )
    .reset_index()
)

income_summary["Good_Loans"] = (
    income_summary["Total_Loans"] -
    income_summary["Bad_Loans"]
)

income_summary["Bad_Rate_Percent"] = (
    income_summary["Bad_Rate"] * 100
)

print("ANNUAL INCOME-WISE CREDIT RISK ANALYSIS")
print("=" * 60)

print(
    income_summary[
        ["Income_Group", "Total_Loans", "Good_Loans",
         "Bad_Loans", "Bad_Rate_Percent"]
    ].round(2).to_string(index=False)
)

# Visualize income-wise bad outcome rates
plt.figure(figsize=(10, 5))

plt.bar(
    income_summary["Income_Group"].astype(str),
    income_summary["Bad_Rate_Percent"]
)

plt.xlabel("Annual Income Group (USD)")
plt.ylabel("Bad Outcome Rate (%)")
plt.title("Credit Risk Across Annual Income Groups")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

del income_analysis


In [ ]:

# Interest rate and credit risk analysis
interest_analysis = pd.DataFrame({
    "int_rate": pd.to_numeric(
        X_train_risk["int_rate"], errors="coerce"
    ),
    "risk": y_train_sample
})

# Define interest rate groups (percentage)
interest_bins = [0, 5, 10, 15, 20, 25, 100]

interest_labels = [
    "Below 5%",
    "5%-10%",
    "10%-15%",
    "15%-20%",
    "20%-25%",
    "Above 25%"
]

interest_analysis["Interest_Group"] = pd.cut(
    interest_analysis["int_rate"],
    bins=interest_bins,
    labels=interest_labels,
    right=False
)

interest_summary = (
    interest_analysis.dropna(subset=["Interest_Group"])
    .groupby("Interest_Group", observed=False)
    .agg(
        Total_Loans=("risk", "count"),
        Bad_Loans=("risk", "sum"),
        Bad_Rate=("risk", "mean")
    )
    .reset_index()
)

interest_summary["Good_Loans"] = (
    interest_summary["Total_Loans"] -
    interest_summary["Bad_Loans"]
)

interest_summary["Bad_Rate_Percent"] = (
    interest_summary["Bad_Rate"] * 100
)

print("INTEREST RATE-WISE CREDIT RISK ANALYSIS")
print("=" * 60)

print(
    interest_summary[
        ["Interest_Group", "Total_Loans", "Good_Loans",
         "Bad_Loans", "Bad_Rate_Percent"]
    ].round(2).to_string(index=False)
)

# Visualize results
plt.figure(figsize=(10, 5))

plt.bar(
    interest_summary["Interest_Group"].astype(str),
    interest_summary["Bad_Rate_Percent"]
)

plt.xlabel("Interest Rate Group")
plt.ylabel("Bad Outcome Rate (%)")
plt.title("Credit Risk Across Interest Rate Groups")
plt.tight_layout()
plt.show()

del interest_analysis


In [ ]:

# Consolidated Banking Portfolio KPIs

total_loans = len(y_train_sample)
good_loans = int((y_train_sample == 0).sum())
bad_loans = int((y_train_sample == 1).sum())

portfolio_bad_rate = bad_loans / total_loans * 100

average_loan_amount = X_train_risk["loan_amnt"].mean()
average_interest_rate = X_train_risk["int_rate"].mean()
average_annual_income = X_train_risk["annual_inc"].mean()
average_dti = X_train_risk["dti"].mean()

portfolio_kpis = pd.DataFrame({
    "Metric": [
        "Total Loans Analyzed",
        "Good Loan Outcomes",
        "Bad Loan Outcomes",
        "Bad Outcome Rate (%)",
        "Average Loan Amount (USD)",
        "Average Interest Rate (%)",
        "Average Annual Income (USD)",
        "Average Debt-to-Income Ratio"
    ],
    "Value": [
        total_loans,
        good_loans,
        bad_loans,
        round(portfolio_bad_rate, 2),
        round(average_loan_amount, 2),
        round(average_interest_rate, 2),
        round(average_annual_income, 2),
        round(average_dti, 2)
    ]
})

print("BANKING PORTFOLIO OVERVIEW")
print("=" * 55)
print(portfolio_kpis.to_string(index=False))

print("\nPORTFOLIO DISTRIBUTION")
print("Good outcomes:", round(good_loans / total_loans * 100, 2), "%")
print("Bad outcomes:", round(bad_loans / total_loans * 100, 2), "%")


In [ ]:

import os
import json
import pandas as pd

# Create project output directory
dashboard_dir = "banking_dashboard_data"
os.makedirs(dashboard_dir, exist_ok=True)

# Export portfolio analytics
analytics_tables = {
    "portfolio_kpis": portfolio_kpis,
    "grade_analysis": grade_summary,
    "purpose_analysis": purpose_summary,
    "income_analysis": income_summary,
    "interest_analysis": interest_summary,
    "feature_importance": importance_df,
    "shap_importance": shap_df
}

for name, dataframe in analytics_tables.items():
    dataframe.to_csv(
        os.path.join(dashboard_dir, f"{name}.csv"),
        index=False
    )

# Save binary model evaluation
binary_metrics = {
    "model": "Weighted CatBoost Binary Classifier",
    "threshold": FINAL_THRESHOLD,
    "accuracy": float(accuracy_score(y_risk_test, test_predictions)),
    "precision": float(precision_score(y_risk_test, test_predictions)),
    "recall": float(recall_score(y_risk_test, test_predictions)),
    "f1_score": float(f1_score(y_risk_test, test_predictions)),
    "roc_auc": float(roc_auc_score(y_risk_test, test_probabilities)),
    "test_records": int(len(y_risk_test)),
    "training_sample_records": int(len(y_train_sample))
}

with open(
    os.path.join(dashboard_dir, "binary_model_metrics.json"),
    "w"
) as file:
    json.dump(binary_metrics, file, indent=4)

print("DASHBOARD DATA EXPORTED SUCCESSFULLY")
print("\nFiles created:")

for filename in sorted(os.listdir(dashboard_dir)):
    print("-", filename)


In [ ]:

import os
import json
import gc

# Create model directory
model_dir = os.path.join(dashboard_dir, "models")
os.makedirs(model_dir, exist_ok=True)

# Save binary credit risk model
binary_model_path = os.path.join(
    model_dir, "binary_credit_risk.cbm"
)
binary_weighted.save_model(binary_model_path)

# Save seven-class loan status model
multiclass_model_path = os.path.join(
    model_dir, "loan_status_multiclass.cbm"
)
weighted_model.save_model(multiclass_model_path)

# Save model metadata
metadata = {
    "project": "AI-Powered Banking and Credit Risk Analysis",
    "binary_model": {
        "file": "binary_credit_risk.cbm",
        "model_type": "CatBoost Binary Classification",
        "target_classes": {
            "0": "Good Outcome",
            "1": "Bad Outcome"
        },
        "prediction_threshold": float(FINAL_THRESHOLD),
        "features": catboost_features
    },
    "multiclass_model": {
        "file": "loan_status_multiclass.cbm",
        "model_type": "CatBoost Multiclass Classification",
        "classes": class_names,
        "features": catboost_features
    }
}

metadata_path = os.path.join(
    model_dir, "model_metadata.json"
)

with open(metadata_path, "w") as file:
    json.dump(metadata, file, indent=4)

gc.collect()

print("MODEL EXPORT COMPLETED")
print("=" * 45)

for filename in sorted(os.listdir(model_dir)):
    path = os.path.join(model_dir, filename)
    print(filename, "-", round(os.path.getsize(path) / (1024**2), 2), "MB")


In [ ]:

from catboost import CatBoostClassifier
import os
import json

# Reload binary model
binary_check = CatBoostClassifier()
binary_check.load_model(
    os.path.join(model_dir, "binary_credit_risk.cbm")
)

# Reload multiclass model
multiclass_check = CatBoostClassifier()
multiclass_check.load_model(
    os.path.join(model_dir, "loan_status_multiclass.cbm")
)

# Load metadata
with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

print("SAVED MODEL VERIFICATION")
print("=" * 45)

print("Binary model loaded:", binary_check.is_fitted())
print("Binary feature count:", binary_check.feature_count_)

print("\nMulticlass model loaded:", multiclass_check.is_fitted())
print("Multiclass feature count:", multiclass_check.feature_count_)

print("\nBinary threshold:",
      saved_metadata["binary_model"]["prediction_threshold"])

print("Multiclass classes:",
      saved_metadata["multiclass_model"]["classes"])

assert binary_check.feature_count_ == 26
assert multiclass_check.feature_count_ == 26

print("\nALL MODEL ARTIFACTS VERIFIED SUCCESSFULLY")


In [ ]:

from catboost import CatBoostClassifier
import os
import json

# Load saved models
binary_check = CatBoostClassifier()
binary_check.load_model(
    os.path.join(model_dir, "binary_credit_risk.cbm")
)

multiclass_check = CatBoostClassifier()
multiclass_check.load_model(
    os.path.join(model_dir, "loan_status_multiclass.cbm")
)

# Load metadata
with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

print("SAVED MODEL VERIFICATION")
print("=" * 45)

print("Binary model loaded:", binary_check.is_fitted())
print("Binary feature count:", binary_check.feature_count())

print("\nMulticlass model loaded:", multiclass_check.is_fitted())
print("Multiclass feature count:", multiclass_check.feature_count())

print("\nBinary threshold:",
      saved_metadata["binary_model"]["prediction_threshold"])

print("Multiclass classes:",
      saved_metadata["multiclass_model"]["classes"])

assert binary_check.feature_count() == 26
assert multiclass_check.feature_count() == 26

print("\nALL MODEL ARTIFACTS VERIFIED SUCCESSFULLY")


In [ ]:

# Use the existing validation sample
binary_test_output = binary_check.predict_proba(
    X_val_risk.iloc[:5]
)

multiclass_test_output = multiclass_check.predict_proba(
    X_val_risk.iloc[:5]
)

print("MODEL VERIFICATION")
print("=" * 45)

print("Binary model fitted:", binary_check.is_fitted())
print("Binary prediction output shape:", binary_test_output.shape)

print("\nMulticlass model fitted:", multiclass_check.is_fitted())
print("Multiclass prediction output shape:", multiclass_test_output.shape)

print("\nFirst 5 binary bad-outcome probabilities:")
print(binary_test_output[:, 1])

print("\nFirst 5 multiclass predictions:")
print(multiclass_check.predict(X_val_risk.iloc[:5]).ravel())

assert binary_test_output.shape == (5, 2)
assert multiclass_test_output.shape == (5, 7)

print("\nBOTH MODELS VERIFIED SUCCESSFULLY")


In [ ]:

import os

print("PROJECT DIRECTORY:", os.getcwd())

print("\nDASHBOARD DATA:")
for root, dirs, files in os.walk(dashboard_dir):
    level = root.replace(dashboard_dir, "").count(os.sep)
    indent = " " * (2 * level)
    print(indent + os.path.basename(root) + "/")
    for filename in files:
        print(indent + "  " + filename)

print("\nSLM CHECKPOINT:")
print(
    r"C:\Users\Manendra\OneDrive\Documents\SLM project\checkpoints\slm_700_iterations.pt"
)

print("\nSLM CHECKPOINT EXISTS:",
    os.path.exists(
        r"C:\Users\Manendra\OneDrive\Documents\SLM project\checkpoints\slm_700_iterations.pt"
    )
)


In [ ]:

import torch
import os

checkpoint_path = (
    r"C:\Users\Manendra\OneDrive\Documents\SLM project"
    r"\checkpoints\slm_700_iterations.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT TYPE:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTOP-LEVEL KEYS:")
    print(list(checkpoint.keys())[:20])

    # Extract model state dictionary
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        state_dict = checkpoint

    print("\nMODEL PARAMETERS")
    print("=" * 50)

    for name, tensor in list(state_dict.items())[:35]:
        if torch.is_tensor(tensor):
            print(name, tuple(tensor.shape))

    qk_keys = [
        name for name in state_dict
        if "q_norm" in name or "k_norm" in name
    ]

    print("\nQK NORMALIZATION CHECK")
    print("QK normalization parameters found:", len(qk_keys))

    for name in qk_keys:
        print(name)
else:
    print("Unexpected checkpoint format.")


In [ ]:
print("SAVED MODEL CONFIGURATION")
print("=" * 50)

for key, value in checkpoint["model_config"].items():
    print(f"{key}: {value}")

print("\nALL MODEL PARAMETER NAMES")
print("=" * 50)

for name in checkpoint["model_state_dict"].keys():
    print(name)

In [ ]:
import inspect

print("MODEL OBJECT EXISTS:", "model" in globals())

if "model" in globals():
    print("\nMODEL CLASS:")
    print(type(model))

    print("\nMODEL ARCHITECTURE SOURCE:")
    print(inspect.getsource(type(model)))
else:
    print("The model object is not currently loaded in this kernel.")

In [ ]:
import torch

print("PYTORCH MODEL OBJECTS")
print("=" * 45)

for name, obj in globals().copy().items():
    if isinstance(obj, torch.nn.Module):
        print(name, "->", type(obj).__name__)